# VIX Multi-Horizon Stacking V2 — Panel étendu (1785 modèles)
## Analyse de corrélation → sélection intelligente → méta-modèle XGBoost OOF
## Horizons : 1j, 2j, 3j, 5j, 7j, 10j | N_features : 5 à 30


In [ ]:
import sys
!{sys.executable} -m pip install -q xgboost lightgbm yfinance pandas_datareader arch pykalman hmmlearn shap xlsxwriter imbalanced-learn statsmodels seaborn

import os, time, json, warnings, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yfinance as yf
import pandas_datareader.data as web
import shap

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, accuracy_score
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("Imports OK")


In [ ]:
CONFIG = {
    'seed': 42, 'start_date': '2000-01-01',
    'flat_thr': 0.003,
    'test_date': None,
    'horizons': [1, 2, 3, 5, 7, 10],
    'meta_n_estimators': 400,
    'meta_max_depth': 4,
    'meta_lr': 0.03,
    'n_folds_oof': 5,
    # Seuils de sélection du panel
    'min_f1_dir': 0.50,          # plancher absolu
    'max_corr_pred': 0.85,       # corrélation max entre prédictions (diversité)
    'top_n_per_regime': 30,      # max modèles par (horizon, regime)
}

TARGET_COL = 'VIX_Amplitude_Class'

YF_TICKERS = """
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX
^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD
AAPL AMZN MSFT NVDA INTC QCOM XOM WMT MCD SBUX
MS COF BLK SCHW CLX CPB LMT NOC GD HON
CCI PSA EQIX NEE TXN PAYX LUV CMCSA
XLK XLF XLE XLV XLU XLB XLI XLY
""".split()

FRED_SERIES = {'NFCI':'NFCI','STLFSI':'STLFSI4','T10Y2Y':'T10Y2Y','EFFR':'EFFR'}

# =============================================================================
# 1785 RUNS issus de tous les Excel (F1_dir >= 0.50, anti-leakage)
# =============================================================================
ALL_RUNS = [
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5787,
  "F1_UP_FORT": 0.2955,
  "F1_DOWN_FORT": 0.3404,
  "Acc_dir": 0.5797,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N6",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5644,
  "F1_UP_FORT": 0.2353,
  "F1_DOWN_FORT": 0.4086,
  "Acc_dir": 0.5652,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N7",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5778,
  "F1_UP_FORT": 0.2,
  "F1_DOWN_FORT": 0.3596,
  "Acc_dir": 0.5797,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N8",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5543,
  "F1_UP_FORT": 0.1333,
  "F1_DOWN_FORT": 0.3529,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N9",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.6027,
  "F1_UP_FORT": 0.1429,
  "F1_DOWN_FORT": 0.3301,
  "Acc_dir": 0.6039,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5926,
  "F1_UP_FORT": 0.1975,
  "F1_DOWN_FORT": 0.3368,
  "Acc_dir": 0.5942,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N11",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5926,
  "F1_UP_FORT": 0.2045,
  "F1_DOWN_FORT": 0.25,
  "Acc_dir": 0.5942,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N12",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.618,
  "F1_UP_FORT": 0.137,
  "F1_DOWN_FORT": 0.2828,
  "Acc_dir": 0.6184,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N13",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 13,
  "F1_dir": 0.5741,
  "F1_UP_FORT": 0.0822,
  "F1_DOWN_FORT": 0.2885,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N14",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 14,
  "F1_dir": 0.5314,
  "F1_UP_FORT": 0.1558,
  "F1_DOWN_FORT": 0.2941,
  "Acc_dir": 0.5314,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N15",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 15,
  "F1_dir": 0.5507,
  "F1_UP_FORT": 0.1333,
  "F1_DOWN_FORT": 0.2941,
  "Acc_dir": 0.5507,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N16",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 16,
  "F1_dir": 0.569,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3232,
  "Acc_dir": 0.57,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 17,
  "F1_dir": 0.5553,
  "F1_UP_FORT": 0.137,
  "F1_DOWN_FORT": 0.3301,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N18",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 18,
  "F1_dir": 0.5699,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3232,
  "Acc_dir": 0.57,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_XGBoost_N19",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 19,
  "F1_dir": 0.5409,
  "F1_UP_FORT": 0.1389,
  "F1_DOWN_FORT": 0.3137,
  "Acc_dir": 0.5411,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d",
   "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5589,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.303,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N6",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5765,
  "F1_UP_FORT": 0.2222,
  "F1_DOWN_FORT": 0.3505,
  "Acc_dir": 0.5797,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N7",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.6277,
  "F1_UP_FORT": 0.1519,
  "F1_DOWN_FORT": 0.34,
  "Acc_dir": 0.628,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N8",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.6027,
  "F1_UP_FORT": 0.1839,
  "F1_DOWN_FORT": 0.3269,
  "Acc_dir": 0.6039,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.596,
  "F1_UP_FORT": 0.2093,
  "F1_DOWN_FORT": 0.3654,
  "Acc_dir": 0.599,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5772,
  "F1_UP_FORT": 0.1687,
  "F1_DOWN_FORT": 0.2737,
  "Acc_dir": 0.5797,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N11",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5797,
  "F1_UP_FORT": 0.1975,
  "F1_DOWN_FORT": 0.25,
  "Acc_dir": 0.5797,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.565,
  "F1_UP_FORT": 0.0811,
  "F1_DOWN_FORT": 0.2887,
  "Acc_dir": 0.5652,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N13",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 13,
  "F1_dir": 0.5651,
  "F1_UP_FORT": 0.1412,
  "F1_DOWN_FORT": 0.2574,
  "Acc_dir": 0.5652,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N14",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 14,
  "F1_dir": 0.5748,
  "F1_UP_FORT": 0.2105,
  "F1_DOWN_FORT": 0.297,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N15",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 15,
  "F1_dir": 0.5584,
  "F1_UP_FORT": 0.125,
  "F1_DOWN_FORT": 0.2947,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 16,
  "F1_dir": 0.5307,
  "F1_UP_FORT": 0.1299,
  "F1_DOWN_FORT": 0.2198,
  "Acc_dir": 0.5314,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N17",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 17,
  "F1_dir": 0.5492,
  "F1_UP_FORT": 0.1408,
  "F1_DOWN_FORT": 0.2558,
  "Acc_dir": 0.5507,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N18",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 18,
  "F1_dir": 0.6135,
  "F1_UP_FORT": 0.2133,
  "F1_DOWN_FORT": 0.3838,
  "Acc_dir": 0.6135,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LightGBM_N19",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 19,
  "F1_dir": 0.5744,
  "F1_UP_FORT": 0.1667,
  "F1_DOWN_FORT": 0.2979,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d",
   "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5694,
  "F1_UP_FORT": 0.2045,
  "F1_DOWN_FORT": 0.3656,
  "Acc_dir": 0.57,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N6",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5744,
  "F1_UP_FORT": 0.2247,
  "F1_DOWN_FORT": 0.3878,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5892,
  "F1_UP_FORT": 0.1707,
  "F1_DOWN_FORT": 0.3956,
  "Acc_dir": 0.5894,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N8",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.6069,
  "F1_UP_FORT": 0.1266,
  "F1_DOWN_FORT": 0.375,
  "Acc_dir": 0.6087,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N9",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.6018,
  "F1_UP_FORT": 0.1882,
  "F1_DOWN_FORT": 0.4043,
  "Acc_dir": 0.6039,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5602,
  "F1_UP_FORT": 0.1905,
  "F1_DOWN_FORT": 0.3261,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5744,
  "F1_UP_FORT": 0.1667,
  "F1_DOWN_FORT": 0.3434,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.565,
  "F1_UP_FORT": 0.1081,
  "F1_DOWN_FORT": 0.2553,
  "Acc_dir": 0.5652,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N13",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 13,
  "F1_dir": 0.5555,
  "F1_UP_FORT": 0.0556,
  "F1_DOWN_FORT": 0.2857,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N14",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 14,
  "F1_dir": 0.5603,
  "F1_UP_FORT": 0.1053,
  "F1_DOWN_FORT": 0.303,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N15",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 15,
  "F1_dir": 0.541,
  "F1_UP_FORT": 0.1067,
  "F1_DOWN_FORT": 0.3429,
  "Acc_dir": 0.5411,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N16",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 16,
  "F1_dir": 0.5726,
  "F1_UP_FORT": 0.2025,
  "F1_DOWN_FORT": 0.3232,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 17,
  "F1_dir": 0.5845,
  "F1_UP_FORT": 0.1143,
  "F1_DOWN_FORT": 0.2917,
  "Acc_dir": 0.5845,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N18",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 18,
  "F1_dir": 0.5555,
  "F1_UP_FORT": 0.1867,
  "F1_DOWN_FORT": 0.3043,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_N19",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 19,
  "F1_dir": 0.57,
  "F1_UP_FORT": 0.1739,
  "F1_DOWN_FORT": 0.2887,
  "Acc_dir": 0.57,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d",
   "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.56,
  "F1_UP_FORT": 0.1842,
  "F1_DOWN_FORT": 0.3784,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N6",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5584,
  "F1_UP_FORT": 0.169,
  "F1_DOWN_FORT": 0.3604,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.6183,
  "F1_UP_FORT": 0.16,
  "F1_DOWN_FORT": 0.4158,
  "Acc_dir": 0.6184,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.56,
  "F1_UP_FORT": 0.1892,
  "F1_DOWN_FORT": 0.42,
  "Acc_dir": 0.5604,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N9",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5981,
  "F1_UP_FORT": 0.2222,
  "F1_DOWN_FORT": 0.3226,
  "Acc_dir": 0.599,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5409,
  "F1_UP_FORT": 0.2716,
  "F1_DOWN_FORT": 0.3125,
  "Acc_dir": 0.5411,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5409,
  "F1_UP_FORT": 0.1867,
  "F1_DOWN_FORT": 0.3148,
  "Acc_dir": 0.5411,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5458,
  "F1_UP_FORT": 0.1892,
  "F1_DOWN_FORT": 0.2885,
  "Acc_dir": 0.5459,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 13,
  "F1_dir": 0.5737,
  "F1_UP_FORT": 0.1449,
  "F1_DOWN_FORT": 0.3186,
  "Acc_dir": 0.5749,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 14,
  "F1_dir": 0.5234,
  "F1_UP_FORT": 0.1918,
  "F1_DOWN_FORT": 0.3214,
  "Acc_dir": 0.5266,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N15",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 15,
  "F1_dir": 0.5501,
  "F1_UP_FORT": 0.2,
  "F1_DOWN_FORT": 0.3333,
  "Acc_dir": 0.5507,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N16",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 16,
  "F1_dir": 0.5553,
  "F1_UP_FORT": 0.2133,
  "F1_DOWN_FORT": 0.3455,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N17",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 17,
  "F1_dir": 0.5941,
  "F1_UP_FORT": 0.1176,
  "F1_DOWN_FORT": 0.3333,
  "Acc_dir": 0.5942,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N18",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 18,
  "F1_dir": 0.5473,
  "F1_UP_FORT": 0.1918,
  "F1_DOWN_FORT": 0.3178,
  "Acc_dir": 0.5507,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_RandomForest_N19",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 19,
  "F1_dir": 0.5623,
  "F1_UP_FORT": 0.1739,
  "F1_DOWN_FORT": 0.3186,
  "Acc_dir": 0.5652,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d",
   "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5446,
  "F1_UP_FORT": 0.1892,
  "F1_DOWN_FORT": 0.3301,
  "Acc_dir": 0.5459,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5697,
  "F1_UP_FORT": 0.1795,
  "F1_DOWN_FORT": 0.3469,
  "Acc_dir": 0.57,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5441,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.3409,
  "Acc_dir": 0.5459,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5344,
  "F1_UP_FORT": 0.2245,
  "F1_DOWN_FORT": 0.3218,
  "Acc_dir": 0.5362,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5331,
  "F1_UP_FORT": 0.2308,
  "F1_DOWN_FORT": 0.3146,
  "Acc_dir": 0.5362,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.542,
  "F1_UP_FORT": 0.2353,
  "F1_DOWN_FORT": 0.2989,
  "Acc_dir": 0.5459,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5543,
  "F1_UP_FORT": 0.28,
  "F1_DOWN_FORT": 0.2955,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5459,
  "F1_UP_FORT": 0.2727,
  "F1_DOWN_FORT": 0.3011,
  "Acc_dir": 0.5459,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N13",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 13,
  "F1_dir": 0.5553,
  "F1_UP_FORT": 0.2921,
  "F1_DOWN_FORT": 0.3441,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N14",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 14,
  "F1_dir": 0.5456,
  "F1_UP_FORT": 0.2889,
  "F1_DOWN_FORT": 0.3226,
  "Acc_dir": 0.5459,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N15",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 15,
  "F1_dir": 0.5553,
  "F1_UP_FORT": 0.2889,
  "F1_DOWN_FORT": 0.3191,
  "Acc_dir": 0.5556,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N16",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 16,
  "F1_dir": 0.5507,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.3061,
  "Acc_dir": 0.5507,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N17",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 17,
  "F1_dir": 0.5383,
  "F1_UP_FORT": 0.225,
  "F1_DOWN_FORT": 0.3301,
  "Acc_dir": 0.5411,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N18",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 18,
  "F1_dir": 0.5338,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.32,
  "Acc_dir": 0.5362,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_LogisticRegression_N19",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 19,
  "F1_dir": 0.5234,
  "F1_UP_FORT": 0.2025,
  "F1_DOWN_FORT": 0.3366,
  "Acc_dir": 0.5266,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d",
   "EOG_EOGResources_ret_5d",
   "VIX_Price_zscore_60d__div__spx_vol_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_5d",
   "ITT_ITTInc_ret_5d",
   "NVDA_vol_20d",
   "hmm_p_stress__zrel__spx_vol_5d",
   "MS_MorganStanley_zscore_60d",
   "STLFSI4_ret_1d__ret5x__VRP",
   "AMGN_Amgen_ret_1d",
   "NFCI_ret_5d",
   "STLFSI4_ret_1d__minus__spx_vol_5d",
   "STLFSI4_ret_1d__macross__BDX_Becton_Dickinson_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_Optuna_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5987,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3711,
  "Acc_dir": 0.599,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_CALM_GradientBoosting_OptunaCal_N7",
  "source": "egarch_v2",
  "algo": "GradientBoostingCal",
  "regime": "CALM",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5942,
  "F1_UP_FORT": 0.1176,
  "F1_DOWN_FORT": 0.3958,
  "Acc_dir": 0.5942,
  "train_start": "2001-08-01",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__ret5x__LLY_ret_1d",
   "hmm_p_stress__minus__spx_vol_5d",
   "XOM_ret_1d",
   "EWQ_France_ret_1d__minus__BDX_Becton_Dickinson_ret_1d",
   "MS_MorganStanley_ret_5d",
   "VIX_Price_zscore_60d__div__VRP",
   "MSTR_Bitcoin3_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5058,
  "F1_UP_FORT": 0.3051,
  "F1_DOWN_FORT": 0.3509,
  "Acc_dir": 0.5122,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5026,
  "F1_UP_FORT": 0.2871,
  "F1_DOWN_FORT": 0.3261,
  "Acc_dir": 0.5065,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d",
   "T_ret_1d__div__BOVESPA_Brazil_ret_1d",
   "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5263,
  "F1_UP_FORT": 0.2971,
  "F1_DOWN_FORT": 0.3461,
  "Acc_dir": 0.5395,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LightGBM_N6",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5268,
  "F1_UP_FORT": 0.2865,
  "F1_DOWN_FORT": 0.3491,
  "Acc_dir": 0.5395,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LightGBM_N7",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5084,
  "F1_UP_FORT": 0.2993,
  "F1_DOWN_FORT": 0.2894,
  "Acc_dir": 0.5122,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.504,
  "F1_UP_FORT": 0.2748,
  "F1_DOWN_FORT": 0.3294,
  "Acc_dir": 0.5079,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d",
   "T_ret_1d__div__BOVESPA_Brazil_ret_1d",
   "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5038,
  "F1_UP_FORT": 0.3007,
  "F1_DOWN_FORT": 0.3325,
  "Acc_dir": 0.5065,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5094,
  "F1_UP_FORT": 0.2693,
  "F1_DOWN_FORT": 0.3367,
  "Acc_dir": 0.5151,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_GradientBoosting_N8",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5226,
  "F1_UP_FORT": 0.2752,
  "F1_DOWN_FORT": 0.319,
  "Acc_dir": 0.5265,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.504,
  "F1_UP_FORT": 0.2637,
  "F1_DOWN_FORT": 0.335,
  "Acc_dir": 0.5079,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d",
   "T_ret_1d__div__BOVESPA_Brazil_ret_1d",
   "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5395,
  "F1_UP_FORT": 0.3167,
  "F1_DOWN_FORT": 0.4103,
  "Acc_dir": 0.5552,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_RandomForest_N6",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5249,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.3883,
  "Acc_dir": 0.5438,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5126,
  "F1_UP_FORT": 0.2663,
  "F1_DOWN_FORT": 0.3983,
  "Acc_dir": 0.5194,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5287,
  "F1_UP_FORT": 0.3021,
  "F1_DOWN_FORT": 0.3864,
  "Acc_dir": 0.5366,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5168,
  "F1_UP_FORT": 0.3089,
  "F1_DOWN_FORT": 0.3967,
  "Acc_dir": 0.5265,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d",
   "T_ret_1d__div__BOVESPA_Brazil_ret_1d",
   "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5265,
  "F1_UP_FORT": 0.267,
  "F1_DOWN_FORT": 0.396,
  "Acc_dir": 0.5265,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5179,
  "F1_UP_FORT": 0.2822,
  "F1_DOWN_FORT": 0.3939,
  "Acc_dir": 0.5179,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5135,
  "F1_UP_FORT": 0.2689,
  "F1_DOWN_FORT": 0.3773,
  "Acc_dir": 0.5136,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5007,
  "F1_UP_FORT": 0.2644,
  "F1_DOWN_FORT": 0.3618,
  "Acc_dir": 0.5007,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5036,
  "F1_UP_FORT": 0.2542,
  "F1_DOWN_FORT": 0.3575,
  "Acc_dir": 0.5036,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d",
   "T_ret_1d__div__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5035,
  "F1_UP_FORT": 0.2682,
  "F1_DOWN_FORT": 0.3526,
  "Acc_dir": 0.5036,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d",
   "T_ret_1d__div__BOVESPA_Brazil_ret_1d",
   "BAC_ret_1d__prod__BOVESPA_Brazil_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_NORMAL_RandomForest_OptunaCal_N8",
  "source": "egarch_v2",
  "algo": "RandomForestCal",
  "regime": "NORMAL",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5034,
  "F1_UP_FORT": 0.2582,
  "F1_DOWN_FORT": 0.2899,
  "Acc_dir": 0.5151,
  "train_start": "2000-11-02",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_var_ev_h1__prod__heston_xi",
   "SBUX_vol_20d",
   "US5Y_Rate_ret_5d",
   "VIX_Price_zscore_60d__div__vix_vol_of_vol_10d",
   "VIX_Price_zscore_60d__minus__XLY_Disc_zscore_60d",
   "EWY_Korea_ret_1d__prod__COF_CapitalOne_ret_5d",
   "XLY_Disc_zscore_60d__prod__BOVESPA_Brazil_ret_1d",
   "PLD_Prologis_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5325,
  "F1_UP_FORT": 0.2182,
  "F1_DOWN_FORT": 0.4963,
  "Acc_dir": 0.561,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N6",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5185,
  "F1_UP_FORT": 0.2078,
  "F1_DOWN_FORT": 0.4762,
  "Acc_dir": 0.5552,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N7",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5023,
  "F1_UP_FORT": 0.2105,
  "F1_DOWN_FORT": 0.4494,
  "Acc_dir": 0.5349,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N8",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5201,
  "F1_UP_FORT": 0.2222,
  "F1_DOWN_FORT": 0.4403,
  "Acc_dir": 0.5494,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N9",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5202,
  "F1_UP_FORT": 0.2208,
  "F1_DOWN_FORT": 0.4552,
  "Acc_dir": 0.5552,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5137,
  "F1_UP_FORT": 0.2297,
  "F1_DOWN_FORT": 0.4539,
  "Acc_dir": 0.5581,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N11",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5404,
  "F1_UP_FORT": 0.2803,
  "F1_DOWN_FORT": 0.493,
  "Acc_dir": 0.5785,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_XGBoost_N12",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5299,
  "F1_UP_FORT": 0.1912,
  "F1_DOWN_FORT": 0.5185,
  "Acc_dir": 0.5843,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5107,
  "F1_UP_FORT": 0.253,
  "F1_DOWN_FORT": 0.4908,
  "Acc_dir": 0.5378,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N6",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5377,
  "F1_UP_FORT": 0.2609,
  "F1_DOWN_FORT": 0.4925,
  "Acc_dir": 0.564,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N7",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5049,
  "F1_UP_FORT": 0.2609,
  "F1_DOWN_FORT": 0.4609,
  "Acc_dir": 0.5291,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N8",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5225,
  "F1_UP_FORT": 0.2532,
  "F1_DOWN_FORT": 0.4377,
  "Acc_dir": 0.5523,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5182,
  "F1_UP_FORT": 0.2771,
  "F1_DOWN_FORT": 0.4528,
  "Acc_dir": 0.5436,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5155,
  "F1_UP_FORT": 0.2683,
  "F1_DOWN_FORT": 0.4667,
  "Acc_dir": 0.5494,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N11",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5428,
  "F1_UP_FORT": 0.2517,
  "F1_DOWN_FORT": 0.4765,
  "Acc_dir": 0.5814,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5491,
  "F1_UP_FORT": 0.2781,
  "F1_DOWN_FORT": 0.5075,
  "Acc_dir": 0.5872,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5076,
  "F1_UP_FORT": 0.242,
  "F1_DOWN_FORT": 0.4669,
  "Acc_dir": 0.5436,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N6",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5304,
  "F1_UP_FORT": 0.239,
  "F1_DOWN_FORT": 0.4803,
  "Acc_dir": 0.564,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5116,
  "F1_UP_FORT": 0.2692,
  "F1_DOWN_FORT": 0.4436,
  "Acc_dir": 0.5407,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N8",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5329,
  "F1_UP_FORT": 0.2945,
  "F1_DOWN_FORT": 0.4385,
  "Acc_dir": 0.5581,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N9",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5521,
  "F1_UP_FORT": 0.2875,
  "F1_DOWN_FORT": 0.4758,
  "Acc_dir": 0.5814,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5672,
  "F1_UP_FORT": 0.2667,
  "F1_DOWN_FORT": 0.4876,
  "Acc_dir": 0.6076,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5241,
  "F1_UP_FORT": 0.2635,
  "F1_DOWN_FORT": 0.4765,
  "Acc_dir": 0.5581,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5631,
  "F1_UP_FORT": 0.2302,
  "F1_DOWN_FORT": 0.4949,
  "Acc_dir": 0.6163,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5298,
  "F1_UP_FORT": 0.1527,
  "F1_DOWN_FORT": 0.5132,
  "Acc_dir": 0.5959,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N6",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.5171,
  "F1_UP_FORT": 0.1,
  "F1_DOWN_FORT": 0.4984,
  "Acc_dir": 0.5959,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5098,
  "F1_UP_FORT": 0.1217,
  "F1_DOWN_FORT": 0.5115,
  "Acc_dir": 0.5756,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5256,
  "F1_UP_FORT": 0.1062,
  "F1_DOWN_FORT": 0.4967,
  "Acc_dir": 0.5872,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N9",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5271,
  "F1_UP_FORT": 0.1197,
  "F1_DOWN_FORT": 0.5096,
  "Acc_dir": 0.5988,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5189,
  "F1_UP_FORT": 0.1111,
  "F1_DOWN_FORT": 0.4935,
  "Acc_dir": 0.5814,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5171,
  "F1_UP_FORT": 0.087,
  "F1_DOWN_FORT": 0.5047,
  "Acc_dir": 0.5959,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5004,
  "F1_UP_FORT": 0.1681,
  "F1_DOWN_FORT": 0.5031,
  "Acc_dir": 0.5843,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5275,
  "F1_UP_FORT": 0.1217,
  "F1_DOWN_FORT": 0.5356,
  "Acc_dir": 0.5785,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.519,
  "F1_UP_FORT": 0.1053,
  "F1_DOWN_FORT": 0.5267,
  "Acc_dir": 0.5756,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.532,
  "F1_UP_FORT": 0.0862,
  "F1_DOWN_FORT": 0.5205,
  "Acc_dir": 0.5843,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5523,
  "F1_UP_FORT": 0.2014,
  "F1_DOWN_FORT": 0.5053,
  "Acc_dir": 0.5872,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 9,
  "F1_dir": 0.5405,
  "F1_UP_FORT": 0.2044,
  "F1_DOWN_FORT": 0.5084,
  "Acc_dir": 0.5901,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 10,
  "F1_dir": 0.5474,
  "F1_UP_FORT": 0.1884,
  "F1_DOWN_FORT": 0.5033,
  "Acc_dir": 0.5988,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 11,
  "F1_dir": 0.5608,
  "F1_UP_FORT": 0.1884,
  "F1_DOWN_FORT": 0.5137,
  "Acc_dir": 0.6017,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5745,
  "F1_UP_FORT": 0.1972,
  "F1_DOWN_FORT": 0.526,
  "Acc_dir": 0.6105,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_Optuna_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5295,
  "F1_UP_FORT": 0.2553,
  "F1_DOWN_FORT": 0.4848,
  "Acc_dir": 0.5785,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_STRESS_LightGBM_OptunaCal_N12",
  "source": "egarch_v2",
  "algo": "LightGBMCal",
  "regime": "STRESS",
  "horizon": 1,
  "n_features": 12,
  "F1_dir": 0.5375,
  "F1_UP_FORT": 0.2838,
  "F1_DOWN_FORT": 0.4842,
  "Acc_dir": 0.5814,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "heston_xi__prod__hmm_p_stress",
   "LOGI_Logitech_ret_1d__prod__CPB_CampbellSoup_ret_1d",
   "HangSeng_HK_vol_20d",
   "heston_var_ev_h7",
   "NFCI_ret_1d__macross__kalman_innovation",
   "XLY_Disc_ret_5d__prod__CPB_CampbellSoup_ret_1d",
   "US3Y_Rate_ret_5d",
   "BA_ret_1d",
   "BLK_BlackRock_zscore_60d",
   "PG_ret_1d__div__vix_vol_of_vol_5d",
   "vix_vol_of_vol_5d__minus__XLY_Disc_ret_5d",
   "SCHW_Schwab_ret_1d__macross__XLY_Disc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h1_GLOBAL_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 1,
  "n_features": 5,
  "F1_dir": 0.5376,
  "F1_UP_FORT": 0.211,
  "F1_DOWN_FORT": 0.4198,
  "Acc_dir": 0.5409,
  "train_start": "2001-02-06",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "heston_xi__div__kalman_innovation",
   "kalman_innovation__zrel__kalman_residual",
   "kalman_innovation__prod__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h1_GLOBAL_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 1,
  "n_features": 6,
  "F1_dir": 0.536,
  "F1_UP_FORT": 0.1992,
  "F1_DOWN_FORT": 0.4223,
  "Acc_dir": 0.5393,
  "train_start": "2001-02-06",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "heston_xi__div__kalman_innovation",
   "kalman_innovation__zrel__kalman_residual",
   "kalman_innovation__prod__kalman_residual",
   "kalman_innovation__minus__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h1_GLOBAL_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 1,
  "n_features": 7,
  "F1_dir": 0.5428,
  "F1_UP_FORT": 0.2055,
  "F1_DOWN_FORT": 0.4152,
  "Acc_dir": 0.5449,
  "train_start": "2001-02-06",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "heston_xi__div__kalman_innovation",
   "kalman_innovation__zrel__kalman_residual",
   "kalman_innovation__prod__kalman_residual",
   "kalman_innovation__minus__kalman_residual",
   "kalman_innovation__macross__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h1_GLOBAL_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 1,
  "n_features": 8,
  "F1_dir": 0.5747,
  "F1_UP_FORT": 0.2218,
  "F1_DOWN_FORT": 0.3916,
  "Acc_dir": 0.5785,
  "train_start": "2001-02-06",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "heston_xi__div__kalman_innovation",
   "kalman_innovation__zrel__kalman_residual",
   "kalman_innovation__prod__kalman_residual",
   "kalman_innovation__minus__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "DIS_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N7",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5151,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3415,
  "Acc_dir": 0.5152,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N8",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5195,
  "F1_UP_FORT": 0.186,
  "F1_DOWN_FORT": 0.3065,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N9",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5193,
  "F1_UP_FORT": 0.2353,
  "F1_DOWN_FORT": 0.3,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N11",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5325,
  "F1_UP_FORT": 0.26,
  "F1_DOWN_FORT": 0.3,
  "Acc_dir": 0.5325,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N12",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5706,
  "F1_UP_FORT": 0.2828,
  "F1_DOWN_FORT": 0.3519,
  "Acc_dir": 0.5714,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N13",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.558,
  "F1_UP_FORT": 0.2947,
  "F1_DOWN_FORT": 0.3214,
  "Acc_dir": 0.5584,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N14",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.552,
  "F1_UP_FORT": 0.3107,
  "F1_DOWN_FORT": 0.3009,
  "Acc_dir": 0.5541,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N15",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.509,
  "F1_UP_FORT": 0.2558,
  "F1_DOWN_FORT": 0.3208,
  "Acc_dir": 0.5108,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N16",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5392,
  "F1_UP_FORT": 0.2529,
  "F1_DOWN_FORT": 0.3063,
  "Acc_dir": 0.5411,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5647,
  "F1_UP_FORT": 0.2247,
  "F1_DOWN_FORT": 0.4118,
  "Acc_dir": 0.5671,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N18",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5136,
  "F1_UP_FORT": 0.2022,
  "F1_DOWN_FORT": 0.3654,
  "Acc_dir": 0.5152,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N19",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5305,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3178,
  "Acc_dir": 0.5325,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_N22",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5172,
  "F1_UP_FORT": 0.0988,
  "F1_DOWN_FORT": 0.3273,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5362,
  "F1_UP_FORT": 0.3119,
  "F1_DOWN_FORT": 0.3636,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N6",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5102,
  "F1_UP_FORT": 0.2316,
  "F1_DOWN_FORT": 0.2832,
  "Acc_dir": 0.5108,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N7",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5107,
  "F1_UP_FORT": 0.2391,
  "F1_DOWN_FORT": 0.2759,
  "Acc_dir": 0.5108,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5003,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.3063,
  "Acc_dir": 0.5022,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5584,
  "F1_UP_FORT": 0.2574,
  "F1_DOWN_FORT": 0.2975,
  "Acc_dir": 0.5584,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N11",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5151,
  "F1_UP_FORT": 0.1628,
  "F1_DOWN_FORT": 0.2703,
  "Acc_dir": 0.5152,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5186,
  "F1_UP_FORT": 0.2553,
  "F1_DOWN_FORT": 0.2703,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N13",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5494,
  "F1_UP_FORT": 0.2766,
  "F1_DOWN_FORT": 0.3333,
  "Acc_dir": 0.5498,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N14",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5273,
  "F1_UP_FORT": 0.2667,
  "F1_DOWN_FORT": 0.3248,
  "Acc_dir": 0.5281,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N15",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5223,
  "F1_UP_FORT": 0.2759,
  "F1_DOWN_FORT": 0.2991,
  "Acc_dir": 0.5238,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5355,
  "F1_UP_FORT": 0.2299,
  "F1_DOWN_FORT": 0.3455,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N17",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5694,
  "F1_UP_FORT": 0.1364,
  "F1_DOWN_FORT": 0.3704,
  "Acc_dir": 0.5714,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N18",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5151,
  "F1_UP_FORT": 0.2588,
  "F1_DOWN_FORT": 0.3148,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N19",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5688,
  "F1_UP_FORT": 0.1882,
  "F1_DOWN_FORT": 0.3364,
  "Acc_dir": 0.5714,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N20",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5123,
  "F1_UP_FORT": 0.2553,
  "F1_DOWN_FORT": 0.3093,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N21",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5682,
  "F1_UP_FORT": 0.2326,
  "F1_DOWN_FORT": 0.3063,
  "Acc_dir": 0.5714,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LightGBM_N22",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5317,
  "F1_UP_FORT": 0.2759,
  "F1_DOWN_FORT": 0.3366,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5365,
  "F1_UP_FORT": 0.233,
  "F1_DOWN_FORT": 0.3761,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5063,
  "F1_UP_FORT": 0.1798,
  "F1_DOWN_FORT": 0.314,
  "Acc_dir": 0.5065,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N8",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5063,
  "F1_UP_FORT": 0.122,
  "F1_DOWN_FORT": 0.2783,
  "Acc_dir": 0.5065,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N9",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5016,
  "F1_UP_FORT": 0.2062,
  "F1_DOWN_FORT": 0.3,
  "Acc_dir": 0.5022,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5231,
  "F1_UP_FORT": 0.2857,
  "F1_DOWN_FORT": 0.2703,
  "Acc_dir": 0.5238,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5438,
  "F1_UP_FORT": 0.297,
  "F1_DOWN_FORT": 0.3704,
  "Acc_dir": 0.5455,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N14",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5253,
  "F1_UP_FORT": 0.2737,
  "F1_DOWN_FORT": 0.3462,
  "Acc_dir": 0.5281,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N16",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5085,
  "F1_UP_FORT": 0.2326,
  "F1_DOWN_FORT": 0.2991,
  "Acc_dir": 0.5108,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5016,
  "F1_UP_FORT": 0.2353,
  "F1_DOWN_FORT": 0.3673,
  "Acc_dir": 0.5065,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N18",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5293,
  "F1_UP_FORT": 0.186,
  "F1_DOWN_FORT": 0.3396,
  "Acc_dir": 0.5325,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N19",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5396,
  "F1_UP_FORT": 0.2697,
  "F1_DOWN_FORT": 0.3462,
  "Acc_dir": 0.5455,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N20",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5253,
  "F1_UP_FORT": 0.2045,
  "F1_DOWN_FORT": 0.3878,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N21",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5309,
  "F1_UP_FORT": 0.2222,
  "F1_DOWN_FORT": 0.3265,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_GradientBoosting_N22",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5348,
  "F1_UP_FORT": 0.1667,
  "F1_DOWN_FORT": 0.3542,
  "Acc_dir": 0.5411,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5538,
  "F1_UP_FORT": 0.1616,
  "F1_DOWN_FORT": 0.3871,
  "Acc_dir": 0.5541,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N6",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5227,
  "F1_UP_FORT": 0.2316,
  "F1_DOWN_FORT": 0.3607,
  "Acc_dir": 0.5238,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5584,
  "F1_UP_FORT": 0.234,
  "F1_DOWN_FORT": 0.3692,
  "Acc_dir": 0.5584,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5627,
  "F1_UP_FORT": 0.2418,
  "F1_DOWN_FORT": 0.3308,
  "Acc_dir": 0.5628,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N9",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5276,
  "F1_UP_FORT": 0.22,
  "F1_DOWN_FORT": 0.2857,
  "Acc_dir": 0.5281,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5536,
  "F1_UP_FORT": 0.2376,
  "F1_DOWN_FORT": 0.2903,
  "Acc_dir": 0.5541,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.567,
  "F1_UP_FORT": 0.2796,
  "F1_DOWN_FORT": 0.3307,
  "Acc_dir": 0.5671,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5751,
  "F1_UP_FORT": 0.2553,
  "F1_DOWN_FORT": 0.3889,
  "Acc_dir": 0.5758,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.541,
  "F1_UP_FORT": 0.2222,
  "F1_DOWN_FORT": 0.3077,
  "Acc_dir": 0.5411,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5404,
  "F1_UP_FORT": 0.2826,
  "F1_DOWN_FORT": 0.3214,
  "Acc_dir": 0.5411,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N16",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5105,
  "F1_UP_FORT": 0.1282,
  "F1_DOWN_FORT": 0.3455,
  "Acc_dir": 0.5108,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N17",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5186,
  "F1_UP_FORT": 0.1429,
  "F1_DOWN_FORT": 0.3529,
  "Acc_dir": 0.5195,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N18",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5054,
  "F1_UP_FORT": 0.1647,
  "F1_DOWN_FORT": 0.3238,
  "Acc_dir": 0.5065,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N20",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5318,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3551,
  "Acc_dir": 0.5325,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N21",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5314,
  "F1_UP_FORT": 0.1687,
  "F1_DOWN_FORT": 0.3519,
  "Acc_dir": 0.5325,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_RandomForest_N22",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5407,
  "F1_UP_FORT": 0.1647,
  "F1_DOWN_FORT": 0.3478,
  "Acc_dir": 0.5411,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5699,
  "F1_UP_FORT": 0.2804,
  "F1_DOWN_FORT": 0.3492,
  "Acc_dir": 0.5758,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5707,
  "F1_UP_FORT": 0.234,
  "F1_DOWN_FORT": 0.3231,
  "Acc_dir": 0.5758,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5611,
  "F1_UP_FORT": 0.2308,
  "F1_DOWN_FORT": 0.3167,
  "Acc_dir": 0.5671,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5755,
  "F1_UP_FORT": 0.26,
  "F1_DOWN_FORT": 0.3115,
  "Acc_dir": 0.5801,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5467,
  "F1_UP_FORT": 0.3243,
  "F1_DOWN_FORT": 0.2881,
  "Acc_dir": 0.5498,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5548,
  "F1_UP_FORT": 0.2632,
  "F1_DOWN_FORT": 0.2857,
  "Acc_dir": 0.5584,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5548,
  "F1_UP_FORT": 0.2549,
  "F1_DOWN_FORT": 0.3063,
  "Acc_dir": 0.5584,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5682,
  "F1_UP_FORT": 0.26,
  "F1_DOWN_FORT": 0.2936,
  "Acc_dir": 0.5714,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N13",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5622,
  "F1_UP_FORT": 0.2069,
  "F1_DOWN_FORT": 0.3167,
  "Acc_dir": 0.5628,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N14",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5661,
  "F1_UP_FORT": 0.2022,
  "F1_DOWN_FORT": 0.3077,
  "Acc_dir": 0.5671,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N15",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5322,
  "F1_UP_FORT": 0.0857,
  "F1_DOWN_FORT": 0.3387,
  "Acc_dir": 0.5325,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N16",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5278,
  "F1_UP_FORT": 0.169,
  "F1_DOWN_FORT": 0.3175,
  "Acc_dir": 0.5281,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N17",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5365,
  "F1_UP_FORT": 0.2133,
  "F1_DOWN_FORT": 0.3607,
  "Acc_dir": 0.5368,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N18",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5404,
  "F1_UP_FORT": 0.2078,
  "F1_DOWN_FORT": 0.35,
  "Acc_dir": 0.5411,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N19",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5442,
  "F1_UP_FORT": 0.16,
  "F1_DOWN_FORT": 0.3248,
  "Acc_dir": 0.5455,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N20",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.579,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.3419,
  "Acc_dir": 0.5801,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N21",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5664,
  "F1_UP_FORT": 0.16,
  "F1_DOWN_FORT": 0.339,
  "Acc_dir": 0.5671,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_LogisticRegression_N22",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5533,
  "F1_UP_FORT": 0.1351,
  "F1_DOWN_FORT": 0.3559,
  "Acc_dir": 0.5541,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "EWM_Malaysia_ret_1d",
   "VIX_Price_zscore_60d__div__Russell_Price_ret_5d",
   "VIX_Price_zscore_60d__minus__BTI_BritishAmerican_ret_5d",
   "QQQ_vol_20d__macross__QQQ_ret_5d",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_Optuna_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.558,
  "F1_UP_FORT": 0.1739,
  "F1_DOWN_FORT": 0.404,
  "Acc_dir": 0.5628,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_CALM_XGBoost_OptunaCal_N17",
  "source": "egarch_v2",
  "algo": "XGBoostCal",
  "regime": "CALM",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5507,
  "F1_UP_FORT": 0.1591,
  "F1_DOWN_FORT": 0.3958,
  "Acc_dir": 0.5541,
  "train_start": "2000-11-02",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "WTI_Oil_FRED_ret_5d__minus__vix_max_abs_ret_5d",
   "NFCI_ret_5d__prod__hmm_p_stress",
   "VIX_Price_zscore_60d__div__MSTR_Bitcoin3_ret_5d",
   "AORD_AUS_ret_5d__minus__spx_drawdown_252d",
   "EWQ_France_zscore_60d",
   "EWQ_France_ret_20d",
   "DHR_ret_1d",
   "MRK_Merck_zscore_60d",
   "LMT_LockheedMartin_vol_20d",
   "XLB_Materials_zscore_60d",
   "QQQ_vol_20d__prod__vix_max_abs_ret_5d",
   "vix_vs_ma20__div__spx_drawdown_252d",
   "vix_vs_ma20__minus__YUM_YumBrands_zscore_60d",
   "IYM_BasicMaterials_ret_20d",
   "QQQ_vol_20d",
   "SCHW_Schwab_ret_5d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5366,
  "F1_UP_FORT": 0.3437,
  "F1_DOWN_FORT": 0.3436,
  "Acc_dir": 0.5368,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N6",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5545,
  "F1_UP_FORT": 0.356,
  "F1_DOWN_FORT": 0.3444,
  "Acc_dir": 0.5548,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N7",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.556,
  "F1_UP_FORT": 0.3526,
  "F1_DOWN_FORT": 0.3745,
  "Acc_dir": 0.5562,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N8",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5311,
  "F1_UP_FORT": 0.3325,
  "F1_DOWN_FORT": 0.3318,
  "Acc_dir": 0.5312,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N9",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5449,
  "F1_UP_FORT": 0.358,
  "F1_DOWN_FORT": 0.3426,
  "Acc_dir": 0.5451,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5062,
  "F1_UP_FORT": 0.3227,
  "F1_DOWN_FORT": 0.307,
  "Acc_dir": 0.5062,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N11",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5529,
  "F1_UP_FORT": 0.3431,
  "F1_DOWN_FORT": 0.343,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N12",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5828,
  "F1_UP_FORT": 0.3713,
  "F1_DOWN_FORT": 0.3606,
  "Acc_dir": 0.5839,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N13",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.567,
  "F1_UP_FORT": 0.3614,
  "F1_DOWN_FORT": 0.3654,
  "Acc_dir": 0.5687,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N14",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.595,
  "F1_UP_FORT": 0.3838,
  "F1_DOWN_FORT": 0.37,
  "Acc_dir": 0.5964,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N15",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5978,
  "F1_UP_FORT": 0.3706,
  "F1_DOWN_FORT": 0.3833,
  "Acc_dir": 0.5992,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N16",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5974,
  "F1_UP_FORT": 0.3536,
  "F1_DOWN_FORT": 0.4126,
  "Acc_dir": 0.5992,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.6053,
  "F1_UP_FORT": 0.3714,
  "F1_DOWN_FORT": 0.4099,
  "Acc_dir": 0.6089,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N18",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5931,
  "F1_UP_FORT": 0.3958,
  "F1_DOWN_FORT": 0.3957,
  "Acc_dir": 0.5992,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N19",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5767,
  "F1_UP_FORT": 0.3473,
  "F1_DOWN_FORT": 0.3915,
  "Acc_dir": 0.5881,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N20",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.593,
  "F1_UP_FORT": 0.3315,
  "F1_DOWN_FORT": 0.4094,
  "Acc_dir": 0.6033,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N21",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5862,
  "F1_UP_FORT": 0.3696,
  "F1_DOWN_FORT": 0.4321,
  "Acc_dir": 0.595,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_XGBoost_N22",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5856,
  "F1_UP_FORT": 0.3694,
  "F1_DOWN_FORT": 0.3973,
  "Acc_dir": 0.5922,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d",
   "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5347,
  "F1_UP_FORT": 0.3206,
  "F1_DOWN_FORT": 0.3516,
  "Acc_dir": 0.5354,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N6",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5244,
  "F1_UP_FORT": 0.3498,
  "F1_DOWN_FORT": 0.3341,
  "Acc_dir": 0.5257,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N7",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5253,
  "F1_UP_FORT": 0.3264,
  "F1_DOWN_FORT": 0.3497,
  "Acc_dir": 0.527,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N8",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5349,
  "F1_UP_FORT": 0.3242,
  "F1_DOWN_FORT": 0.3442,
  "Acc_dir": 0.5368,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5497,
  "F1_UP_FORT": 0.3526,
  "F1_DOWN_FORT": 0.3519,
  "Acc_dir": 0.552,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5367,
  "F1_UP_FORT": 0.3558,
  "F1_DOWN_FORT": 0.3441,
  "Acc_dir": 0.5368,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N11",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5582,
  "F1_UP_FORT": 0.3469,
  "F1_DOWN_FORT": 0.3624,
  "Acc_dir": 0.5603,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5649,
  "F1_UP_FORT": 0.2995,
  "F1_DOWN_FORT": 0.3415,
  "Acc_dir": 0.5673,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N13",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5739,
  "F1_UP_FORT": 0.3562,
  "F1_DOWN_FORT": 0.3493,
  "Acc_dir": 0.5756,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N14",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5635,
  "F1_UP_FORT": 0.3402,
  "F1_DOWN_FORT": 0.3801,
  "Acc_dir": 0.5673,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N15",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5928,
  "F1_UP_FORT": 0.3679,
  "F1_DOWN_FORT": 0.3829,
  "Acc_dir": 0.595,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5798,
  "F1_UP_FORT": 0.3378,
  "F1_DOWN_FORT": 0.3934,
  "Acc_dir": 0.5839,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N17",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.6,
  "F1_UP_FORT": 0.3616,
  "F1_DOWN_FORT": 0.3881,
  "Acc_dir": 0.6047,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N18",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5846,
  "F1_UP_FORT": 0.3464,
  "F1_DOWN_FORT": 0.3732,
  "Acc_dir": 0.5936,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N19",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5653,
  "F1_UP_FORT": 0.3371,
  "F1_DOWN_FORT": 0.4093,
  "Acc_dir": 0.577,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N20",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5865,
  "F1_UP_FORT": 0.3257,
  "F1_DOWN_FORT": 0.4148,
  "Acc_dir": 0.5978,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N21",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5878,
  "F1_UP_FORT": 0.2989,
  "F1_DOWN_FORT": 0.4339,
  "Acc_dir": 0.6006,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LightGBM_N22",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5792,
  "F1_UP_FORT": 0.3343,
  "F1_DOWN_FORT": 0.4115,
  "Acc_dir": 0.5908,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d",
   "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5505,
  "F1_UP_FORT": 0.3364,
  "F1_DOWN_FORT": 0.3548,
  "Acc_dir": 0.5506,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N6",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5645,
  "F1_UP_FORT": 0.3823,
  "F1_DOWN_FORT": 0.3632,
  "Acc_dir": 0.5645,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.549,
  "F1_UP_FORT": 0.3276,
  "F1_DOWN_FORT": 0.3401,
  "Acc_dir": 0.5492,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N8",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5505,
  "F1_UP_FORT": 0.3512,
  "F1_DOWN_FORT": 0.3597,
  "Acc_dir": 0.5506,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N9",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.541,
  "F1_UP_FORT": 0.3105,
  "F1_DOWN_FORT": 0.3029,
  "Acc_dir": 0.5423,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5532,
  "F1_UP_FORT": 0.33,
  "F1_DOWN_FORT": 0.3576,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5722,
  "F1_UP_FORT": 0.355,
  "F1_DOWN_FORT": 0.3382,
  "Acc_dir": 0.5728,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5726,
  "F1_UP_FORT": 0.3594,
  "F1_DOWN_FORT": 0.3547,
  "Acc_dir": 0.5742,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N13",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5638,
  "F1_UP_FORT": 0.3487,
  "F1_DOWN_FORT": 0.362,
  "Acc_dir": 0.5659,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N14",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.582,
  "F1_UP_FORT": 0.3724,
  "F1_DOWN_FORT": 0.3973,
  "Acc_dir": 0.5839,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N15",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5931,
  "F1_UP_FORT": 0.3673,
  "F1_DOWN_FORT": 0.3982,
  "Acc_dir": 0.595,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N16",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.6023,
  "F1_UP_FORT": 0.3717,
  "F1_DOWN_FORT": 0.3864,
  "Acc_dir": 0.6047,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.6123,
  "F1_UP_FORT": 0.3686,
  "F1_DOWN_FORT": 0.4161,
  "Acc_dir": 0.6172,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N18",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5894,
  "F1_UP_FORT": 0.3516,
  "F1_DOWN_FORT": 0.3949,
  "Acc_dir": 0.5978,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N19",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5828,
  "F1_UP_FORT": 0.3579,
  "F1_DOWN_FORT": 0.4063,
  "Acc_dir": 0.5908,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N20",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5967,
  "F1_UP_FORT": 0.3575,
  "F1_DOWN_FORT": 0.4167,
  "Acc_dir": 0.6075,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N21",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.6038,
  "F1_UP_FORT": 0.3397,
  "F1_DOWN_FORT": 0.4051,
  "Acc_dir": 0.613,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_N22",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.6082,
  "F1_UP_FORT": 0.3558,
  "F1_DOWN_FORT": 0.4298,
  "Acc_dir": 0.6172,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d",
   "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5362,
  "F1_UP_FORT": 0.3229,
  "F1_DOWN_FORT": 0.3542,
  "Acc_dir": 0.5395,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N6",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5532,
  "F1_UP_FORT": 0.3117,
  "F1_DOWN_FORT": 0.3696,
  "Acc_dir": 0.5562,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5475,
  "F1_UP_FORT": 0.2994,
  "F1_DOWN_FORT": 0.3761,
  "Acc_dir": 0.552,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5635,
  "F1_UP_FORT": 0.328,
  "F1_DOWN_FORT": 0.3919,
  "Acc_dir": 0.5673,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N9",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5596,
  "F1_UP_FORT": 0.3055,
  "F1_DOWN_FORT": 0.384,
  "Acc_dir": 0.5631,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5497,
  "F1_UP_FORT": 0.3037,
  "F1_DOWN_FORT": 0.393,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5645,
  "F1_UP_FORT": 0.3407,
  "F1_DOWN_FORT": 0.3947,
  "Acc_dir": 0.57,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5762,
  "F1_UP_FORT": 0.3526,
  "F1_DOWN_FORT": 0.3877,
  "Acc_dir": 0.5825,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5623,
  "F1_UP_FORT": 0.3095,
  "F1_DOWN_FORT": 0.3878,
  "Acc_dir": 0.5673,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5627,
  "F1_UP_FORT": 0.3351,
  "F1_DOWN_FORT": 0.4048,
  "Acc_dir": 0.5714,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N15",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5758,
  "F1_UP_FORT": 0.3226,
  "F1_DOWN_FORT": 0.409,
  "Acc_dir": 0.5853,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N16",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5618,
  "F1_UP_FORT": 0.3298,
  "F1_DOWN_FORT": 0.4041,
  "Acc_dir": 0.5687,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N17",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5822,
  "F1_UP_FORT": 0.3416,
  "F1_DOWN_FORT": 0.396,
  "Acc_dir": 0.5895,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N18",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5701,
  "F1_UP_FORT": 0.3342,
  "F1_DOWN_FORT": 0.4206,
  "Acc_dir": 0.5798,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N19",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5585,
  "F1_UP_FORT": 0.3116,
  "F1_DOWN_FORT": 0.4165,
  "Acc_dir": 0.5742,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N20",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5604,
  "F1_UP_FORT": 0.314,
  "F1_DOWN_FORT": 0.4311,
  "Acc_dir": 0.577,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N21",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5712,
  "F1_UP_FORT": 0.3218,
  "F1_DOWN_FORT": 0.4355,
  "Acc_dir": 0.5867,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_RandomForest_N22",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.564,
  "F1_UP_FORT": 0.317,
  "F1_DOWN_FORT": 0.4181,
  "Acc_dir": 0.5784,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d",
   "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5548,
  "F1_UP_FORT": 0.3789,
  "F1_DOWN_FORT": 0.404,
  "Acc_dir": 0.5548,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.552,
  "F1_UP_FORT": 0.3729,
  "F1_DOWN_FORT": 0.4131,
  "Acc_dir": 0.552,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5409,
  "F1_UP_FORT": 0.3697,
  "F1_DOWN_FORT": 0.4152,
  "Acc_dir": 0.5409,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.5368,
  "F1_UP_FORT": 0.3723,
  "F1_DOWN_FORT": 0.4031,
  "Acc_dir": 0.5368,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5512,
  "F1_UP_FORT": 0.42,
  "F1_DOWN_FORT": 0.4115,
  "Acc_dir": 0.552,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5596,
  "F1_UP_FORT": 0.4337,
  "F1_DOWN_FORT": 0.3947,
  "Acc_dir": 0.5603,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5572,
  "F1_UP_FORT": 0.4055,
  "F1_DOWN_FORT": 0.352,
  "Acc_dir": 0.5576,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5407,
  "F1_UP_FORT": 0.4,
  "F1_DOWN_FORT": 0.3353,
  "Acc_dir": 0.5409,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N13",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5378,
  "F1_UP_FORT": 0.3909,
  "F1_DOWN_FORT": 0.3235,
  "Acc_dir": 0.5381,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N14",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5436,
  "F1_UP_FORT": 0.386,
  "F1_DOWN_FORT": 0.3516,
  "Acc_dir": 0.5437,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N15",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5532,
  "F1_UP_FORT": 0.3991,
  "F1_DOWN_FORT": 0.3516,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N16",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5345,
  "F1_UP_FORT": 0.3498,
  "F1_DOWN_FORT": 0.3626,
  "Acc_dir": 0.5354,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N17",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5388,
  "F1_UP_FORT": 0.3685,
  "F1_DOWN_FORT": 0.3503,
  "Acc_dir": 0.5395,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N18",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5436,
  "F1_UP_FORT": 0.391,
  "F1_DOWN_FORT": 0.3641,
  "Acc_dir": 0.5437,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N19",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5381,
  "F1_UP_FORT": 0.3532,
  "F1_DOWN_FORT": 0.3914,
  "Acc_dir": 0.5381,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N20",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 20,
  "F1_dir": 0.5534,
  "F1_UP_FORT": 0.3632,
  "F1_DOWN_FORT": 0.4192,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N21",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 21,
  "F1_dir": 0.5534,
  "F1_UP_FORT": 0.3632,
  "F1_DOWN_FORT": 0.414,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_LogisticRegression_N22",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 22,
  "F1_dir": 0.5534,
  "F1_UP_FORT": 0.3632,
  "F1_DOWN_FORT": 0.4122,
  "Acc_dir": 0.5534,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d",
   "VIX_Price_zscore_60d__div__heston_var_ev_h3",
   "US3M_Rate_zscore_60d",
   "kalman_filtered__minus__XLY_Disc_zscore_60d",
   "vix_zscore_10d__macross__MSFT_ret_5d",
   "VOD_Vodafone_vol_20d__macross__COF_CapitalOne_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_Optuna_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5848,
  "F1_UP_FORT": 0.3627,
  "F1_DOWN_FORT": 0.383,
  "Acc_dir": 0.5881,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_NORMAL_GradientBoosting_OptunaCal_N17",
  "source": "egarch_v2",
  "algo": "GradientBoostingCal",
  "regime": "NORMAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5584,
  "F1_UP_FORT": 0.1418,
  "F1_DOWN_FORT": 0.3878,
  "Acc_dir": 0.5687,
  "train_start": "2001-02-20",
  "sampler": "SMOTETomek",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "vix_zscore_10d__div__heston_var_ev_h1",
   "NFCI_ret_5d__zrel__NEE_NextEra_ret_5d",
   "VIX_Price_zscore_60d__zrel__heston_var_ev_h3",
   "HangSeng_HK_ret_5d",
   "VIX_Price_zscore_60d__ret5x__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__div__COF_CapitalOne_ret_5d",
   "DUK_Duke_ret_5d__ret5x__COF_CapitalOne_ret_5d",
   "EWL_Switzerland_vol_20d",
   "EWM_Malaysia_vol_20d",
   "EOG_EOGResources_vol_20d",
   "kalman_filtered__prod__VOD_Vodafone_vol_20d",
   "MSFT_ret_5d__div__NEE_NextEra_ret_5d",
   "GD_GeneralDynamics_zscore_60d",
   "ENB_EnbridgeInc_ret_1d",
   "INTC_ret_5d",
   "MSFT_ret_5d__zrel__AVB_AvalonBay_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5095,
  "F1_UP_FORT": 0.2889,
  "F1_DOWN_FORT": 0.4046,
  "Acc_dir": 0.5419,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N6",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5068,
  "F1_UP_FORT": 0.1931,
  "F1_DOWN_FORT": 0.4769,
  "Acc_dir": 0.5698,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N7",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.513,
  "F1_UP_FORT": 0.1831,
  "F1_DOWN_FORT": 0.484,
  "Acc_dir": 0.5782,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5108,
  "F1_UP_FORT": 0.2439,
  "F1_DOWN_FORT": 0.4361,
  "Acc_dir": 0.5587,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N11",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5345,
  "F1_UP_FORT": 0.2767,
  "F1_DOWN_FORT": 0.4632,
  "Acc_dir": 0.595,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N12",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5281,
  "F1_UP_FORT": 0.2805,
  "F1_DOWN_FORT": 0.4818,
  "Acc_dir": 0.5838,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N13",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5484,
  "F1_UP_FORT": 0.3114,
  "F1_DOWN_FORT": 0.4873,
  "Acc_dir": 0.595,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N14",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5613,
  "F1_UP_FORT": 0.3077,
  "F1_DOWN_FORT": 0.4765,
  "Acc_dir": 0.6089,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N15",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.556,
  "F1_UP_FORT": 0.2917,
  "F1_DOWN_FORT": 0.4702,
  "Acc_dir": 0.6145,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N16",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.549,
  "F1_UP_FORT": 0.3097,
  "F1_DOWN_FORT": 0.4674,
  "Acc_dir": 0.6006,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5536,
  "F1_UP_FORT": 0.3205,
  "F1_DOWN_FORT": 0.4795,
  "Acc_dir": 0.6089,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_XGBoost_N18",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5345,
  "F1_UP_FORT": 0.28,
  "F1_DOWN_FORT": 0.4846,
  "Acc_dir": 0.5922,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d",
   "ORCL_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5362,
  "F1_UP_FORT": 0.236,
  "F1_DOWN_FORT": 0.4576,
  "Acc_dir": 0.5866,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N11",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5447,
  "F1_UP_FORT": 0.2785,
  "F1_DOWN_FORT": 0.5,
  "Acc_dir": 0.595,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.568,
  "F1_UP_FORT": 0.3214,
  "F1_DOWN_FORT": 0.4571,
  "Acc_dir": 0.6173,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N13",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5622,
  "F1_UP_FORT": 0.3145,
  "F1_DOWN_FORT": 0.4884,
  "Acc_dir": 0.6173,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N14",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5648,
  "F1_UP_FORT": 0.3354,
  "F1_DOWN_FORT": 0.4781,
  "Acc_dir": 0.6089,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N15",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5297,
  "F1_UP_FORT": 0.2345,
  "F1_DOWN_FORT": 0.4739,
  "Acc_dir": 0.6006,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5406,
  "F1_UP_FORT": 0.2642,
  "F1_DOWN_FORT": 0.4643,
  "Acc_dir": 0.5922,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N17",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5617,
  "F1_UP_FORT": 0.3394,
  "F1_DOWN_FORT": 0.471,
  "Acc_dir": 0.6117,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LightGBM_N18",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5144,
  "F1_UP_FORT": 0.2313,
  "F1_DOWN_FORT": 0.4605,
  "Acc_dir": 0.5866,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d",
   "ORCL_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5224,
  "F1_UP_FORT": 0.3169,
  "F1_DOWN_FORT": 0.424,
  "Acc_dir": 0.5475,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N6",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.5112,
  "F1_UP_FORT": 0.1757,
  "F1_DOWN_FORT": 0.4794,
  "Acc_dir": 0.567,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.509,
  "F1_UP_FORT": 0.1769,
  "F1_DOWN_FORT": 0.4565,
  "Acc_dir": 0.5698,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5387,
  "F1_UP_FORT": 0.2338,
  "F1_DOWN_FORT": 0.4355,
  "Acc_dir": 0.595,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5517,
  "F1_UP_FORT": 0.2649,
  "F1_DOWN_FORT": 0.4604,
  "Acc_dir": 0.6117,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5493,
  "F1_UP_FORT": 0.2981,
  "F1_DOWN_FORT": 0.4519,
  "Acc_dir": 0.6034,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N13",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5302,
  "F1_UP_FORT": 0.2517,
  "F1_DOWN_FORT": 0.4487,
  "Acc_dir": 0.5922,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N14",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5443,
  "F1_UP_FORT": 0.2963,
  "F1_DOWN_FORT": 0.4521,
  "Acc_dir": 0.5922,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N15",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5408,
  "F1_UP_FORT": 0.2548,
  "F1_DOWN_FORT": 0.4532,
  "Acc_dir": 0.595,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N16",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5236,
  "F1_UP_FORT": 0.2466,
  "F1_DOWN_FORT": 0.4484,
  "Acc_dir": 0.5894,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5281,
  "F1_UP_FORT": 0.2468,
  "F1_DOWN_FORT": 0.4514,
  "Acc_dir": 0.5866,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_GradientBoosting_N18",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5089,
  "F1_UP_FORT": 0.2297,
  "F1_DOWN_FORT": 0.4702,
  "Acc_dir": 0.5726,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d",
   "ORCL_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.529,
  "F1_UP_FORT": 0.2857,
  "F1_DOWN_FORT": 0.4706,
  "Acc_dir": 0.5726,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5094,
  "F1_UP_FORT": 0.1167,
  "F1_DOWN_FORT": 0.4969,
  "Acc_dir": 0.5866,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5485,
  "F1_UP_FORT": 0.3537,
  "F1_DOWN_FORT": 0.4735,
  "Acc_dir": 0.5866,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5552,
  "F1_UP_FORT": 0.3667,
  "F1_DOWN_FORT": 0.5205,
  "Acc_dir": 0.595,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5359,
  "F1_UP_FORT": 0.3234,
  "F1_DOWN_FORT": 0.5051,
  "Acc_dir": 0.5838,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5635,
  "F1_UP_FORT": 0.3333,
  "F1_DOWN_FORT": 0.52,
  "Acc_dir": 0.6117,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5564,
  "F1_UP_FORT": 0.3832,
  "F1_DOWN_FORT": 0.5288,
  "Acc_dir": 0.6006,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N15",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5721,
  "F1_UP_FORT": 0.3657,
  "F1_DOWN_FORT": 0.5306,
  "Acc_dir": 0.6061,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N16",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5516,
  "F1_UP_FORT": 0.3793,
  "F1_DOWN_FORT": 0.5133,
  "Acc_dir": 0.5866,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N17",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.543,
  "F1_UP_FORT": 0.3832,
  "F1_DOWN_FORT": 0.5084,
  "Acc_dir": 0.5838,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_N18",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5406,
  "F1_UP_FORT": 0.3137,
  "F1_DOWN_FORT": 0.5223,
  "Acc_dir": 0.5922,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d",
   "ORCL_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5234,
  "F1_UP_FORT": 0.3708,
  "F1_DOWN_FORT": 0.493,
  "Acc_dir": 0.5503,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5425,
  "F1_UP_FORT": 0.3696,
  "F1_DOWN_FORT": 0.4875,
  "Acc_dir": 0.5642,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5437,
  "F1_UP_FORT": 0.3607,
  "F1_DOWN_FORT": 0.4875,
  "Acc_dir": 0.5642,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N13",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5484,
  "F1_UP_FORT": 0.3626,
  "F1_DOWN_FORT": 0.4946,
  "Acc_dir": 0.5698,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N14",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5483,
  "F1_UP_FORT": 0.3804,
  "F1_DOWN_FORT": 0.4895,
  "Acc_dir": 0.5726,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N15",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5519,
  "F1_UP_FORT": 0.3892,
  "F1_DOWN_FORT": 0.4876,
  "Acc_dir": 0.5754,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N16",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5472,
  "F1_UP_FORT": 0.3978,
  "F1_DOWN_FORT": 0.4823,
  "Acc_dir": 0.5698,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N17",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5437,
  "F1_UP_FORT": 0.3936,
  "F1_DOWN_FORT": 0.4823,
  "Acc_dir": 0.567,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_LogisticRegression_N18",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.519,
  "F1_UP_FORT": 0.3656,
  "F1_DOWN_FORT": 0.4638,
  "Acc_dir": 0.5363,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress",
   "PFE_ret_1d",
   "ENB_EnbridgeInc_ret_5d__macross__CMCSA_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__AMD_zscore_60d",
   "ORCL_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_Optuna_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5539,
  "F1_UP_FORT": 0.3598,
  "F1_DOWN_FORT": 0.5106,
  "Acc_dir": 0.5894,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_STRESS_RandomForest_OptunaCal_N14",
  "source": "egarch_v2",
  "algo": "RandomForestCal",
  "regime": "STRESS",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5497,
  "F1_UP_FORT": 0.3404,
  "F1_DOWN_FORT": 0.4981,
  "Acc_dir": 0.6341,
  "train_start": "2000-11-15",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "vix_mean_abs_ret_5d__prod__heston_xi",
   "Core_PCE_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__MCD_ret_1d",
   "VIX_Price_zscore_60d__macross__AMD_zscore_60d",
   "ENB_EnbridgeInc_ret_5d__zrel__CMCSA_zscore_60d",
   "US7Y_Rate_ret_20d",
   "JNJ_ret_1d",
   "EQIX_Equinix_ret_5d",
   "TM_Telephone_ret_1d",
   "CVX_ret_20d__zrel__NFCI_ret_5d",
   "heston_xi__div__hmm_p_stress",
   "ENB_EnbridgeInc_ret_5d__div__hmm_p_stress",
   "COST_ret_5d__div__vix_vs_ma20",
   "hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.6197,
  "F1_UP_FORT": 0.3818,
  "F1_DOWN_FORT": 0.4664,
  "Acc_dir": 0.6229,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.6015,
  "F1_UP_FORT": 0.362,
  "F1_DOWN_FORT": 0.4407,
  "Acc_dir": 0.6038,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.6789,
  "F1_UP_FORT": 0.5024,
  "F1_DOWN_FORT": 0.5444,
  "Acc_dir": 0.6802,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LightGBM_N14",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.6796,
  "F1_UP_FORT": 0.5245,
  "F1_DOWN_FORT": 0.5528,
  "Acc_dir": 0.6802,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.6765,
  "F1_UP_FORT": 0.5325,
  "F1_DOWN_FORT": 0.5549,
  "Acc_dir": 0.6771,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d",
   "vix_vol_of_vol_10d__div__kalman_residual",
   "heston_xi__div__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.6124,
  "F1_UP_FORT": 0.3846,
  "F1_DOWN_FORT": 0.4643,
  "Acc_dir": 0.6145,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.6169,
  "F1_UP_FORT": 0.3962,
  "F1_DOWN_FORT": 0.4656,
  "Acc_dir": 0.6206,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.675,
  "F1_UP_FORT": 0.4615,
  "F1_DOWN_FORT": 0.5248,
  "Acc_dir": 0.6779,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_RandomForest_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.6721,
  "F1_UP_FORT": 0.4706,
  "F1_DOWN_FORT": 0.5314,
  "Acc_dir": 0.674,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 5,
  "F1_dir": 0.5655,
  "F1_UP_FORT": 0.2104,
  "F1_DOWN_FORT": 0.4251,
  "Acc_dir": 0.5733,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 6,
  "F1_dir": 0.57,
  "F1_UP_FORT": 0.1871,
  "F1_DOWN_FORT": 0.4307,
  "Acc_dir": 0.5779,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 7,
  "F1_dir": 0.5716,
  "F1_UP_FORT": 0.1839,
  "F1_DOWN_FORT": 0.4498,
  "Acc_dir": 0.5794,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 8,
  "F1_dir": 0.556,
  "F1_UP_FORT": 0.2895,
  "F1_DOWN_FORT": 0.4389,
  "Acc_dir": 0.5595,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 9,
  "F1_dir": 0.5658,
  "F1_UP_FORT": 0.2676,
  "F1_DOWN_FORT": 0.4424,
  "Acc_dir": 0.5687,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 10,
  "F1_dir": 0.5642,
  "F1_UP_FORT": 0.284,
  "F1_DOWN_FORT": 0.4334,
  "Acc_dir": 0.5664,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 11,
  "F1_dir": 0.5718,
  "F1_UP_FORT": 0.2832,
  "F1_DOWN_FORT": 0.4258,
  "Acc_dir": 0.574,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 12,
  "F1_dir": 0.5575,
  "F1_UP_FORT": 0.2947,
  "F1_DOWN_FORT": 0.428,
  "Acc_dir": 0.5595,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N13",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 13,
  "F1_dir": 0.5669,
  "F1_UP_FORT": 0.3178,
  "F1_DOWN_FORT": 0.4264,
  "Acc_dir": 0.5679,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N14",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 14,
  "F1_dir": 0.5567,
  "F1_UP_FORT": 0.3034,
  "F1_DOWN_FORT": 0.4138,
  "Acc_dir": 0.5573,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N15",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 15,
  "F1_dir": 0.5529,
  "F1_UP_FORT": 0.3216,
  "F1_DOWN_FORT": 0.4166,
  "Acc_dir": 0.5534,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d",
   "vix_vol_of_vol_10d__div__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N16",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 16,
  "F1_dir": 0.5529,
  "F1_UP_FORT": 0.3236,
  "F1_DOWN_FORT": 0.4144,
  "Acc_dir": 0.5534,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d",
   "vix_vol_of_vol_10d__div__kalman_residual",
   "heston_xi__div__kalman_residual"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N17",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 17,
  "F1_dir": 0.5558,
  "F1_UP_FORT": 0.3255,
  "F1_DOWN_FORT": 0.4186,
  "Acc_dir": 0.5565,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d",
   "vix_vol_of_vol_10d__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "spx_momentum_3d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N18",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 18,
  "F1_dir": 0.5594,
  "F1_UP_FORT": 0.3196,
  "F1_DOWN_FORT": 0.4327,
  "Acc_dir": 0.5603,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d",
   "vix_vol_of_vol_10d__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "spx_momentum_3d",
   "vix_zscore_10d__zrel__XLB_Materials_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h3_GLOBAL_LogisticRegression_N19",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "GLOBAL",
  "horizon": 3,
  "n_features": 19,
  "F1_dir": 0.5616,
  "F1_UP_FORT": 0.3265,
  "F1_DOWN_FORT": 0.4408,
  "Acc_dir": 0.5626,
  "train_start": "2007-01-31",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "kalman_innovation__div__kalman_residual",
   "kalman_innovation__macross__kalman_residual",
   "PAYX_Paychex_vol_20d",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "TXN_vol_20d__div__heston_xi",
   "TXN_vol_20d__div__kalman_residual",
   "CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__minus__NFCI_zscore_60d",
   "vix_zscore_10d__minus__EWQ_France_zscore_60d",
   "AMZN_ret_5d",
   "ITT_ITTInc_ret_5d",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWL_Switzerland_vol_20d",
   "VVIX_ret_20d",
   "vix_vol_of_vol_10d__div__kalman_residual",
   "heston_xi__div__kalman_residual",
   "spx_momentum_3d",
   "vix_zscore_10d__zrel__XLB_Materials_zscore_60d",
   "VIX_Price_zscore_60d__ret5x__HangSeng_HK_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 5,
  "F1_dir": 0.5169,
  "F1_UP_FORT": 0.3396,
  "F1_DOWN_FORT": 0.3876,
  "Acc_dir": 0.5172,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N8",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5083,
  "F1_UP_FORT": 0.2712,
  "F1_DOWN_FORT": 0.3103,
  "Acc_dir": 0.5086,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N9",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5255,
  "F1_UP_FORT": 0.2075,
  "F1_DOWN_FORT": 0.381,
  "Acc_dir": 0.5259,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5257,
  "F1_UP_FORT": 0.2373,
  "F1_DOWN_FORT": 0.3846,
  "Acc_dir": 0.5259,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N11",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5301,
  "F1_UP_FORT": 0.193,
  "F1_DOWN_FORT": 0.3511,
  "Acc_dir": 0.5302,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N12",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5039,
  "F1_UP_FORT": 0.1042,
  "F1_DOWN_FORT": 0.4088,
  "Acc_dir": 0.5043,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N13",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5517,
  "F1_UP_FORT": 0.1818,
  "F1_DOWN_FORT": 0.4088,
  "Acc_dir": 0.5517,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N14",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 14,
  "F1_dir": 0.5602,
  "F1_UP_FORT": 0.1136,
  "F1_DOWN_FORT": 0.3971,
  "Acc_dir": 0.5603,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N15",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 15,
  "F1_dir": 0.5723,
  "F1_UP_FORT": 0.1616,
  "F1_DOWN_FORT": 0.3898,
  "Acc_dir": 0.5733,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N16",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5637,
  "F1_UP_FORT": 0.1684,
  "F1_DOWN_FORT": 0.3361,
  "Acc_dir": 0.5647,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5904,
  "F1_UP_FORT": 0.1765,
  "F1_DOWN_FORT": 0.3299,
  "Acc_dir": 0.5948,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_XGBoost_N18",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.5626,
  "F1_UP_FORT": 0.1538,
  "F1_DOWN_FORT": 0.2692,
  "Acc_dir": 0.569,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d",
   "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5577,
  "F1_UP_FORT": 0.25,
  "F1_DOWN_FORT": 0.4034,
  "Acc_dir": 0.5603,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N11",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5241,
  "F1_UP_FORT": 0.2479,
  "F1_DOWN_FORT": 0.3415,
  "Acc_dir": 0.5259,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5472,
  "F1_UP_FORT": 0.0833,
  "F1_DOWN_FORT": 0.3438,
  "Acc_dir": 0.5474,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N13",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5546,
  "F1_UP_FORT": 0.1458,
  "F1_DOWN_FORT": 0.3969,
  "Acc_dir": 0.556,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N14",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 14,
  "F1_dir": 0.5118,
  "F1_UP_FORT": 0.1489,
  "F1_DOWN_FORT": 0.3538,
  "Acc_dir": 0.5129,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N15",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 15,
  "F1_dir": 0.5469,
  "F1_UP_FORT": 0.1224,
  "F1_DOWN_FORT": 0.3359,
  "Acc_dir": 0.5517,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5342,
  "F1_UP_FORT": 0.1042,
  "F1_DOWN_FORT": 0.3443,
  "Acc_dir": 0.5388,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N17",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5722,
  "F1_UP_FORT": 0.125,
  "F1_DOWN_FORT": 0.3168,
  "Acc_dir": 0.5776,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_N18",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.5207,
  "F1_UP_FORT": 0.1333,
  "F1_DOWN_FORT": 0.2752,
  "Acc_dir": 0.5259,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d",
   "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N5",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 5,
  "F1_dir": 0.5263,
  "F1_UP_FORT": 0.3273,
  "F1_DOWN_FORT": 0.4194,
  "Acc_dir": 0.5302,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N9",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5317,
  "F1_UP_FORT": 0.2373,
  "F1_DOWN_FORT": 0.2931,
  "Acc_dir": 0.5345,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5645,
  "F1_UP_FORT": 0.2783,
  "F1_DOWN_FORT": 0.3333,
  "Acc_dir": 0.5647,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5328,
  "F1_UP_FORT": 0.2143,
  "F1_DOWN_FORT": 0.3279,
  "Acc_dir": 0.5345,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5474,
  "F1_UP_FORT": 0.1616,
  "F1_DOWN_FORT": 0.4328,
  "Acc_dir": 0.5474,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N13",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5582,
  "F1_UP_FORT": 0.26,
  "F1_DOWN_FORT": 0.3651,
  "Acc_dir": 0.5603,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N14",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 14,
  "F1_dir": 0.5554,
  "F1_UP_FORT": 0.1935,
  "F1_DOWN_FORT": 0.4091,
  "Acc_dir": 0.556,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N15",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 15,
  "F1_dir": 0.5837,
  "F1_UP_FORT": 0.1429,
  "F1_DOWN_FORT": 0.3802,
  "Acc_dir": 0.5862,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N16",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5657,
  "F1_UP_FORT": 0.1443,
  "F1_DOWN_FORT": 0.3361,
  "Acc_dir": 0.569,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5508,
  "F1_UP_FORT": 0.1165,
  "F1_DOWN_FORT": 0.2804,
  "Acc_dir": 0.556,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_GradientBoosting_N18",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.5669,
  "F1_UP_FORT": 0.1333,
  "F1_DOWN_FORT": 0.339,
  "Acc_dir": 0.569,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d",
   "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 7,
  "F1_dir": 0.5073,
  "F1_UP_FORT": 0.2264,
  "F1_DOWN_FORT": 0.3165,
  "Acc_dir": 0.5086,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5068,
  "F1_UP_FORT": 0.2308,
  "F1_DOWN_FORT": 0.3597,
  "Acc_dir": 0.5086,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N9",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5301,
  "F1_UP_FORT": 0.2,
  "F1_DOWN_FORT": 0.3438,
  "Acc_dir": 0.5302,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5776,
  "F1_UP_FORT": 0.2883,
  "F1_DOWN_FORT": 0.3846,
  "Acc_dir": 0.5776,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5689,
  "F1_UP_FORT": 0.2727,
  "F1_DOWN_FORT": 0.3664,
  "Acc_dir": 0.569,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5143,
  "F1_UP_FORT": 0.2105,
  "F1_DOWN_FORT": 0.3433,
  "Acc_dir": 0.5172,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5176,
  "F1_UP_FORT": 0.1739,
  "F1_DOWN_FORT": 0.3741,
  "Acc_dir": 0.5216,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N15",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 15,
  "F1_dir": 0.5342,
  "F1_UP_FORT": 0.2364,
  "F1_DOWN_FORT": 0.3485,
  "Acc_dir": 0.5345,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N16",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5258,
  "F1_UP_FORT": 0.2018,
  "F1_DOWN_FORT": 0.3761,
  "Acc_dir": 0.5259,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N17",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5688,
  "F1_UP_FORT": 0.2857,
  "F1_DOWN_FORT": 0.386,
  "Acc_dir": 0.569,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_RandomForest_N18",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.556,
  "F1_UP_FORT": 0.2807,
  "F1_DOWN_FORT": 0.3729,
  "Acc_dir": 0.556,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d",
   "Michigan_Sentiment_zscore_60d__zrel__AMZN_zscore_60d",
   "NFCI_ret_5d__div__SO_SouthernCo_ret_1d",
   "XLY_Disc_vol_20d",
   "ASML_ASML_ret_5d__div__spx_momentum_3d",
   "NFCI_ret_5d",
   "Nikkei_Japan_zscore_60d__prod__AMZN_zscore_60d",
   "T10Y2Y_Spread_ret_5d",
   "TM_Telephone_vol_20d",
   "vix_zscore_10d__minus__HON_Honeywell_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 6,
  "F1_dir": 0.5016,
  "F1_UP_FORT": 0.2222,
  "F1_DOWN_FORT": 0.3425,
  "Acc_dir": 0.5043,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_Optuna_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5571,
  "F1_UP_FORT": 0.1964,
  "F1_DOWN_FORT": 0.3964,
  "Acc_dir": 0.5603,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_CALM_LightGBM_OptunaCal_N9",
  "source": "egarch_v2",
  "algo": "LightGBMCal",
  "regime": "CALM",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5611,
  "F1_UP_FORT": 0.1964,
  "F1_DOWN_FORT": 0.4,
  "Acc_dir": 0.5647,
  "train_start": "2000-11-15",
  "sampler": "SMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__minus__HON_Honeywell_zscore_60d",
   "NFCI_ret_5d__div__NVDA_vol_20d",
   "vix_zscore_10d__ret5x__SO_SouthernCo_ret_1d",
   "VIX_Price_zscore_60d__div__VRP",
   "vix_zscore_10d__div__spx_abs_ret_max_5d",
   "EWY_Korea_ret_20d",
   "EWQ_France_zscore_60d",
   "LLY_zscore_60d",
   "EWC_Canada_zscore_60d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N5",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 5,
  "F1_dir": 0.5007,
  "F1_UP_FORT": 0.34,
  "F1_DOWN_FORT": 0.3119,
  "Acc_dir": 0.5021,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N6",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 6,
  "F1_dir": 0.5061,
  "F1_UP_FORT": 0.2783,
  "F1_DOWN_FORT": 0.3293,
  "Acc_dir": 0.5062,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N7",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 7,
  "F1_dir": 0.5138,
  "F1_UP_FORT": 0.33,
  "F1_DOWN_FORT": 0.343,
  "Acc_dir": 0.5144,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N8",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5451,
  "F1_UP_FORT": 0.2797,
  "F1_DOWN_FORT": 0.4116,
  "Acc_dir": 0.5528,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N9",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5289,
  "F1_UP_FORT": 0.2757,
  "F1_DOWN_FORT": 0.4258,
  "Acc_dir": 0.5377,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N10",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5164,
  "F1_UP_FORT": 0.2588,
  "F1_DOWN_FORT": 0.4208,
  "Acc_dir": 0.5295,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N16",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5098,
  "F1_UP_FORT": 0.2157,
  "F1_DOWN_FORT": 0.439,
  "Acc_dir": 0.5281,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N17",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5185,
  "F1_UP_FORT": 0.2459,
  "F1_DOWN_FORT": 0.4467,
  "Acc_dir": 0.5309,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N18",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.5229,
  "F1_UP_FORT": 0.2324,
  "F1_DOWN_FORT": 0.4282,
  "Acc_dir": 0.535,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_XGBoost_N19",
  "source": "egarch_v2",
  "algo": "XGBoost",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 19,
  "F1_dir": 0.5299,
  "F1_UP_FORT": 0.2147,
  "F1_DOWN_FORT": 0.4444,
  "Acc_dir": 0.5446,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d",
   "vix_max_abs_ret_5d__macross__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N5",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 5,
  "F1_dir": 0.5111,
  "F1_UP_FORT": 0.3644,
  "F1_DOWN_FORT": 0.3003,
  "Acc_dir": 0.5117,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N6",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 6,
  "F1_dir": 0.5032,
  "F1_UP_FORT": 0.2851,
  "F1_DOWN_FORT": 0.3427,
  "Acc_dir": 0.5034,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N8",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5253,
  "F1_UP_FORT": 0.2768,
  "F1_DOWN_FORT": 0.4068,
  "Acc_dir": 0.5364,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N9",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5289,
  "F1_UP_FORT": 0.2835,
  "F1_DOWN_FORT": 0.4149,
  "Acc_dir": 0.5377,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N10",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5192,
  "F1_UP_FORT": 0.248,
  "F1_DOWN_FORT": 0.3855,
  "Acc_dir": 0.5322,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N12",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5075,
  "F1_UP_FORT": 0.2209,
  "F1_DOWN_FORT": 0.4181,
  "Acc_dir": 0.5281,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N13",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5121,
  "F1_UP_FORT": 0.226,
  "F1_DOWN_FORT": 0.4396,
  "Acc_dir": 0.5336,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N16",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5232,
  "F1_UP_FORT": 0.2254,
  "F1_DOWN_FORT": 0.4115,
  "Acc_dir": 0.5391,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N17",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5192,
  "F1_UP_FORT": 0.2597,
  "F1_DOWN_FORT": 0.4145,
  "Acc_dir": 0.5322,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N18",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.5276,
  "F1_UP_FORT": 0.2611,
  "F1_DOWN_FORT": 0.4289,
  "Acc_dir": 0.5364,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LightGBM_N19",
  "source": "egarch_v2",
  "algo": "LightGBM",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 19,
  "F1_dir": 0.5143,
  "F1_UP_FORT": 0.2305,
  "F1_DOWN_FORT": 0.4526,
  "Acc_dir": 0.5322,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d",
   "vix_max_abs_ret_5d__macross__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N6",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 6,
  "F1_dir": 0.5226,
  "F1_UP_FORT": 0.2991,
  "F1_DOWN_FORT": 0.2701,
  "Acc_dir": 0.5226,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N7",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 7,
  "F1_dir": 0.5224,
  "F1_UP_FORT": 0.289,
  "F1_DOWN_FORT": 0.2997,
  "Acc_dir": 0.5226,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N8",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5188,
  "F1_UP_FORT": 0.2698,
  "F1_DOWN_FORT": 0.3552,
  "Acc_dir": 0.524,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N9",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5133,
  "F1_UP_FORT": 0.2567,
  "F1_DOWN_FORT": 0.4203,
  "Acc_dir": 0.5199,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N10",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.512,
  "F1_UP_FORT": 0.234,
  "F1_DOWN_FORT": 0.4167,
  "Acc_dir": 0.5295,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N11",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5138,
  "F1_UP_FORT": 0.1905,
  "F1_DOWN_FORT": 0.4174,
  "Acc_dir": 0.5322,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N12",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5075,
  "F1_UP_FORT": 0.2393,
  "F1_DOWN_FORT": 0.4428,
  "Acc_dir": 0.5267,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N13",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5015,
  "F1_UP_FORT": 0.2273,
  "F1_DOWN_FORT": 0.4615,
  "Acc_dir": 0.5267,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N15",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 15,
  "F1_dir": 0.5167,
  "F1_UP_FORT": 0.2543,
  "F1_DOWN_FORT": 0.4363,
  "Acc_dir": 0.5377,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N16",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.537,
  "F1_UP_FORT": 0.2626,
  "F1_DOWN_FORT": 0.4329,
  "Acc_dir": 0.5487,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N17",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5133,
  "F1_UP_FORT": 0.2949,
  "F1_DOWN_FORT": 0.4232,
  "Acc_dir": 0.524,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N18",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.5098,
  "F1_UP_FORT": 0.2575,
  "F1_DOWN_FORT": 0.459,
  "Acc_dir": 0.524,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_GradientBoosting_N19",
  "source": "egarch_v2",
  "algo": "GradientBoosting",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 19,
  "F1_dir": 0.5374,
  "F1_UP_FORT": 0.2466,
  "F1_DOWN_FORT": 0.4661,
  "Acc_dir": 0.5487,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d",
   "vix_max_abs_ret_5d__macross__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N5",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 5,
  "F1_dir": 0.5044,
  "F1_UP_FORT": 0.3394,
  "F1_DOWN_FORT": 0.3641,
  "Acc_dir": 0.5048,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N6",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 6,
  "F1_dir": 0.5322,
  "F1_UP_FORT": 0.3106,
  "F1_DOWN_FORT": 0.3478,
  "Acc_dir": 0.5322,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N7",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 7,
  "F1_dir": 0.5267,
  "F1_UP_FORT": 0.3306,
  "F1_DOWN_FORT": 0.3711,
  "Acc_dir": 0.5267,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N8",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5635,
  "F1_UP_FORT": 0.2835,
  "F1_DOWN_FORT": 0.4449,
  "Acc_dir": 0.5665,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N9",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5547,
  "F1_UP_FORT": 0.272,
  "F1_DOWN_FORT": 0.4458,
  "Acc_dir": 0.561,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N10",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5594,
  "F1_UP_FORT": 0.2644,
  "F1_DOWN_FORT": 0.4575,
  "Acc_dir": 0.5693,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N11",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5539,
  "F1_UP_FORT": 0.2388,
  "F1_DOWN_FORT": 0.4689,
  "Acc_dir": 0.5652,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N12",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5515,
  "F1_UP_FORT": 0.2655,
  "F1_DOWN_FORT": 0.4694,
  "Acc_dir": 0.5693,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N13",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5403,
  "F1_UP_FORT": 0.2955,
  "F1_DOWN_FORT": 0.4664,
  "Acc_dir": 0.5556,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N14",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 14,
  "F1_dir": 0.541,
  "F1_UP_FORT": 0.2651,
  "F1_DOWN_FORT": 0.4664,
  "Acc_dir": 0.5569,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N15",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 15,
  "F1_dir": 0.5518,
  "F1_UP_FORT": 0.3236,
  "F1_DOWN_FORT": 0.4686,
  "Acc_dir": 0.5652,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N16",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 16,
  "F1_dir": 0.5559,
  "F1_UP_FORT": 0.2772,
  "F1_DOWN_FORT": 0.4589,
  "Acc_dir": 0.5679,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N17",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 17,
  "F1_dir": 0.5502,
  "F1_UP_FORT": 0.2873,
  "F1_DOWN_FORT": 0.4686,
  "Acc_dir": 0.561,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N18",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 18,
  "F1_dir": 0.559,
  "F1_UP_FORT": 0.3133,
  "F1_DOWN_FORT": 0.4711,
  "Acc_dir": 0.5693,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_RandomForest_N19",
  "source": "egarch_v2",
  "algo": "RandomForest",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 19,
  "F1_dir": 0.5534,
  "F1_UP_FORT": 0.2695,
  "F1_DOWN_FORT": 0.4564,
  "Acc_dir": 0.5665,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP",
   "VIX_Price_zscore_60d__div__MCD_ret_5d",
   "SO_SouthernCo_ret_5d",
   "vix_vol_of_vol_10d__prod__heston_theta",
   "heston_var_ev_h1__minus__MCD_ret_5d",
   "EMR_Emerson_ret_20d",
   "vix_max_abs_ret_5d__macross__heston_theta"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N5",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 5,
  "F1_dir": 0.5212,
  "F1_UP_FORT": 0.3556,
  "F1_DOWN_FORT": 0.3762,
  "Acc_dir": 0.5213,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N6",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 6,
  "F1_dir": 0.5391,
  "F1_UP_FORT": 0.3381,
  "F1_DOWN_FORT": 0.3758,
  "Acc_dir": 0.5391,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N7",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 7,
  "F1_dir": 0.5295,
  "F1_UP_FORT": 0.3192,
  "F1_DOWN_FORT": 0.3815,
  "Acc_dir": 0.5295,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N8",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 8,
  "F1_dir": 0.5473,
  "F1_UP_FORT": 0.3014,
  "F1_DOWN_FORT": 0.4146,
  "Acc_dir": 0.5487,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N9",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 9,
  "F1_dir": 0.5498,
  "F1_UP_FORT": 0.3095,
  "F1_DOWN_FORT": 0.4087,
  "Acc_dir": 0.5514,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N10",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 10,
  "F1_dir": 0.5324,
  "F1_UP_FORT": 0.3243,
  "F1_DOWN_FORT": 0.4077,
  "Acc_dir": 0.5336,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N11",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 11,
  "F1_dir": 0.5375,
  "F1_UP_FORT": 0.3187,
  "F1_DOWN_FORT": 0.4426,
  "Acc_dir": 0.5391,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N12",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 12,
  "F1_dir": 0.5398,
  "F1_UP_FORT": 0.2877,
  "F1_DOWN_FORT": 0.4229,
  "Acc_dir": 0.5418,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d"
  ]
 },
 {
  "model_id": "egarch_v2_h5_NORMAL_LogisticRegression_N13",
  "source": "egarch_v2",
  "algo": "LogisticRegression",
  "regime": "NORMAL",
  "horizon": 5,
  "n_features": 13,
  "F1_dir": 0.5398,
  "F1_UP_FORT": 0.2808,
  "F1_DOWN_FORT": 0.4306,
  "Acc_dir": 0.5418,
  "train_start": "2001-02-06",
  "sampler": "BorderlineSMOTE",
  "best_params": "{}",
  "features": [
   "VIX_Price_zscore_60d__prod__CCI_CrownCastle_vol_20d",
   "VIX_Price_zscore_60d__div__heston_theta",
   "VIX_Price_zscore_60d__div__hmm_p_stress",
   "EWY_Korea_ret_20d",
   "M_Macys_vol_20d",
   "Core_CPI_zscore_60d__macross__VRP",
   "VIX_Price_zscore_60d__div__vix_max_abs_ret_5d",
   "NFCI_ret_5d",
   "vix_zscore_10d__div__vix_vol_of_vol_10d",
   "Nikkei_Japan_vol_20d",
   "AMT_AmericanTower_zscore_60d__minus__vix_level",
   "3M_vol_20d",
   "VIX_Price_zscore_60d__div__VRP"
  ]
 }
]  # premiers 500 pour la démo

print(f"{len(ALL_RUNS)} runs chargés")
df_runs = pd.DataFrame(ALL_RUNS)
print(df_runs.groupby(['horizon','regime'])['model_id'].count().unstack(fill_value=0).to_string())


In [ ]:
def load_data(start=CONFIG['start_date']):
    t0 = time.time()
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^','IDX_').replace('-','_') for c in raw.columns]
    raw = raw.loc[:, raw.notna().mean()>=0.90].ffill().dropna(how='all')
    fred_frames = []
    for name,sid in FRED_SERIES.items():
        try:
            s=web.DataReader(sid,'fred',start).squeeze(); s.name=f'FRED_{name}'
            fred_frames.append(s)
        except Exception as e: print(f"  [WARN] {e}")
    if fred_frames:
        raw=pd.concat([raw,pd.concat(fred_frames,axis=1).reindex(raw.index,method='ffill')],axis=1)
    print(f"  {raw.shape[0]}j × {raw.shape[1]} séries ({time.time()-t0:.1f}s)")
    return raw

df_raw   = load_data()
all_dates= df_raw.dropna(how='all').index.sort_values()
split_idx= int(len(all_dates)*0.80)
TEST_DATE= all_dates[split_idx].strftime('%Y-%m-%d')
CONFIG['test_date'] = TEST_DATE
print(f"Split 80/20: train→{all_dates[split_idx-1].date()} | test→{all_dates[split_idx].date()}")


In [ ]:
# Copier ici les fonctions build_all_features() et build_target()
# depuis VIX_MULTIHORIZON_STACKING.ipynb (cellules 4 et 5)
# ainsi que reconstruct_feature(), build_feature_matrix(),
# get_algo(), get_sampler(), compute_metrics()

# Après copie :
vix_col = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX'))
            and 'VXN' not in c and 'VVIX' not in c][0]
print(f"[INFO] Copier les fonctions depuis VIX_MULTIHORIZON_STACKING.ipynb cellules 4-7")
print(f"[INFO] Puis exécuter build_all_features() et build_target()")


## Analyse exploratoire du panel

Avant de sélectionner les modèles pour le stacking, on analyse :
1. Distribution des F1_dir par horizon/régime/algo/N_features
2. Corrélation entre N_features et performance
3. Identification des configurations prometteuses


In [ ]:
# =============================================================================
# ANALYSE EXPLORATOIRE — Panel de 1785 runs
# =============================================================================
df_runs = pd.DataFrame(ALL_RUNS)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Distribution F1_dir par horizon
ax = axes[0,0]
for h in sorted(df_runs['horizon'].unique()):
    d = df_runs[df_runs['horizon']==h]['F1_dir']
    ax.hist(d, bins=20, alpha=0.6, label=f'h={h}j', density=True)
ax.set_title('Distribution F1_dir par horizon'); ax.legend(); ax.set_xlabel('F1_dir')

# 2. F1_dir médian par (horizon, regime)
ax = axes[0,1]
pivot = df_runs.groupby(['horizon','regime'])['F1_dir'].median().unstack(fill_value=0)
pivot.plot(kind='bar', ax=ax, colormap='viridis')
ax.set_title('F1_dir médian par horizon × régime'); ax.set_xlabel('Horizon')
ax.legend(loc='upper right', fontsize=8)

# 3. F1_dir vs N_features
ax = axes[0,2]
for regime in ['CALM','NORMAL','STRESS','GLOBAL']:
    sub = df_runs[df_runs['regime']==regime]
    mn = sub.groupby('n_features')['F1_dir'].mean()
    ax.plot(mn.index, mn.values, marker='o', markersize=3, label=regime, alpha=0.8)
ax.set_title('F1_dir moyen vs N_features'); ax.set_xlabel('N_features')
ax.legend(); ax.axhline(0.55, color='red', linestyle='--', alpha=0.5, label='0.55')

# 4. Distribution par algo
ax = axes[1,0]
df_runs.groupby('algo')['F1_dir'].median().sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('F1_dir médian par algorithme'); ax.axvline(0.55, color='red', linestyle='--')

# 5. Heatmap horizon × N_features (F1_dir médian)
ax = axes[1,1]
hm = df_runs.groupby(['horizon','n_features'])['F1_dir'].median().unstack(fill_value=0)
sns.heatmap(hm, ax=ax, cmap='YlOrRd', annot=False, fmt='.2f')
ax.set_title('F1_dir médian : horizon × N_features')

# 6. F1_UP_FORT vs F1_DOWN_FORT scatter
ax = axes[1,2]
for algo in df_runs['algo'].unique():
    sub = df_runs[df_runs['algo']==algo]
    ax.scatter(sub['F1_UP_FORT'], sub['F1_DOWN_FORT'], alpha=0.3, s=10, label=algo)
ax.set_xlabel('F1_UP_FORT'); ax.set_ylabel('F1_DOWN_FORT')
ax.set_title('F1_UP_FORT vs F1_DOWN_FORT'); ax.legend(fontsize=7)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('panel_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : panel_analysis.png")

# Stats descriptives
print("\n=== Statistiques descriptives ===")
print(df_runs.groupby('algo')[['F1_dir','F1_UP_FORT','F1_DOWN_FORT']].agg(['mean','std','max']).round(4).to_string())
print("\n=== Corrélations avec F1_dir ===")
num_cols = ['n_features','F1_dir','F1_UP_FORT','F1_DOWN_FORT','Acc_dir']
corr = df_runs[[c for c in num_cols if c in df_runs.columns]].corr()
print(corr['F1_dir'].sort_values(ascending=False).to_string())


## Sélection intelligente du panel

Critères de sélection en 3 étapes :
1. **Plancher performance** : F1_dir ≥ 0.50 (déjà filtré)
2. **Diversité** : éliminer les modèles dont les prédictions sont trop corrélées entre elles (> 0.85) — garder le plus performant de chaque cluster
3. **Équilibre** : limiter à `top_n_per_regime` par (horizon, régime) pour éviter la surreprésentation


In [ ]:
# =============================================================================
# SÉLECTION DU PANEL PAR DIVERSITÉ
# =============================================================================
def select_diverse_panel(df_runs, trained_proba_dict, max_corr=0.85, top_n=30):
    """
    Sélectionne un panel diversifié de modèles.

    Algorithme greedy :
    1. Trier par F1_dir décroissant
    2. Ajouter le premier modèle
    3. Pour chaque modèle suivant : calculer la corrélation de ses prédictions
       avec tous les modèles déjà sélectionnés
    4. Ajouter uniquement si la corrélation max < max_corr

    Si les prédictions ne sont pas encore disponibles (avant entraînement),
    utiliser la corrélation des vecteurs de features comme proxy.
    """
    selected = []
    selected_proba = []

    df_sorted = df_runs.sort_values('F1_dir', ascending=False)

    for _, row in df_sorted.iterrows():
        mid = row['model_id']

        if mid not in trained_proba_dict:
            # Fallback : sélection par diversité de features
            if len(selected) == 0:
                selected.append(mid); continue
            # Vérifier le overlap de features avec les sélectionnés
            feats_new = set(row['features'])
            max_overlap = max(
                len(feats_new & set(df_runs.loc[df_runs['model_id']==s,'features'].iloc[0]))
                / max(len(feats_new), 1)
                for s in selected
            ) if selected else 0
            if max_overlap < 0.80:
                selected.append(mid)
        else:
            proba_new = trained_proba_dict[mid]
            if len(selected_proba) == 0:
                selected.append(mid); selected_proba.append(proba_new); continue
            # Direction binaire pour la corrélation
            dir_new = (proba_new[:, 2] + proba_new[:, 3] > 0.5).astype(int)
            corrs = [np.corrcoef(dir_new,
                                  (p[:,2]+p[:,3]>0.5).astype(int))[0,1]
                     for p in selected_proba]
            if max(corrs) < max_corr:
                selected.append(mid); selected_proba.append(proba_new)

        if len(selected) >= top_n: break

    return selected


# Sélection préliminaire basée sur les features (avant entraînement)
panel_by_config = {}
for h in sorted(df_runs['horizon'].unique()):
    for reg in ['CALM','NORMAL','STRESS','GLOBAL']:
        sub = df_runs[(df_runs['horizon']==h) & (df_runs['regime']==reg)].copy()
        if sub.empty: continue
        selected = select_diverse_panel(sub, {}, max_corr=0.80,
                                         top_n=CONFIG['top_n_per_regime'])
        panel_by_config[(h,reg)] = selected
        print(f"  h={h}j {reg:<8}: {len(sub)} runs → {len(selected)} sélectionnés | "
              f"F1 range [{sub.loc[sub['model_id'].isin(selected),'F1_dir'].min():.3f}"
              f" - {sub.loc[sub['model_id'].isin(selected),'F1_dir'].max():.3f}]")

all_selected_ids = list(set(mid for ids in panel_by_config.values() for mid in ids))
df_panel = df_runs[df_runs['model_id'].isin(all_selected_ids)].copy()
print(f"\nPanel final : {len(df_panel)} modèles sélectionnés sur {len(df_runs)}")


In [ ]:
# =============================================================================
# ENTRAÎNEMENT DU PANEL SÉLECTIONNÉ
# =============================================================================
trained_models   = {}
model_scalers    = {}
model_features   = {}
model_metrics    = {}
model_proba_test = {}

t_total = time.time()

# Targets pour tous les horizons (1j existants + 2j et 10j nouveaux)
targets_by_h = {}
for h in CONFIG['horizons']:
    tgt, reg_s, thr = build_target(df_features[vix_col], h, split_idx, df_features)
    targets_by_h[h] = {'target': tgt, 'regime': reg_s, 'thresholds': thr}

def train_single_model(cfg, df_features, targets_by_h, vix_col):
    """Entraîne un modèle du panel et retourne les prédictions test."""
    model_id = cfg['model_id']
    h        = cfg['horizon']
    regime   = cfg['regime']
    algo     = cfg['algo']
    features = cfg['features']
    ts       = cfg['train_start']

    tgt_info = targets_by_h[h]
    target   = tgt_info['target']
    reg_s    = tgt_info['regime']

    X_mat = build_feature_matrix(df_features.reindex(target.index), features)
    if X_mat.empty: return None, None, None, None, None
    X_mat[TARGET_COL] = target

    df_tr = X_mat.loc[(X_mat.index < pd.Timestamp(TEST_DATE)) &
                       (X_mat.index >= pd.Timestamp(ts))].dropna(subset=[TARGET_COL])
    df_te = X_mat.loc[X_mat.index >= pd.Timestamp(TEST_DATE)].dropna(subset=[TARGET_COL])

    if regime != 'GLOBAL':
        df_tr = df_tr.loc[reg_s.reindex(df_tr.index)==regime]
        df_te = df_te.loc[reg_s.reindex(df_te.index)==regime]

    feat_cols = [c for c in X_mat.columns if c!=TARGET_COL]
    if len(df_tr)<30 or len(df_te)<5: return None, None, None, None, None

    y_tr = df_tr[TARGET_COL].values.astype(int)
    y_te = df_te[TARGET_COL].values.astype(int)
    X_tr = df_tr[feat_cols].fillna(0).values
    X_te = df_te[feat_cols].fillna(0).values

    sc = RobustScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    try:
        samp = get_sampler(cfg.get('sampler','SMOTE'))
        X_res, y_res = samp.fit_resample(X_tr_sc, y_tr)
    except: X_res, y_res = X_tr_sc, y_tr

    try:
        clf = get_algo(algo, len(feat_cols), cfg.get('best_params','{}'))
        clf.fit(X_res, y_res)
    except: return None, None, None, None, None

    y_pred = clf.predict(X_te_sc)
    y_prob = clf.predict_proba(X_te_sc)
    if y_prob.shape[1] < 4:
        fp = np.zeros((len(y_prob),4))
        for ci,c in enumerate(clf.classes_): fp[:,int(c)] = y_prob[:,ci]
        y_prob = fp

    met = compute_metrics(y_te, y_pred, y_prob)
    return clf, sc, feat_cols, y_prob, met


n_total = len(df_panel)
for i, (_, cfg) in enumerate(df_panel.iterrows()):
    mid = cfg['model_id']
    t0  = time.time()
    clf, sc, feat_cols, y_prob, met = train_single_model(dict(cfg), df_features, targets_by_h, vix_col)
    if clf is None: continue

    trained_models[mid]   = clf
    model_scalers[mid]    = sc
    model_features[mid]   = feat_cols
    model_metrics[mid]    = {**met, 'horizon':cfg['horizon'],'regime':cfg['regime'],
                              'algo':cfg['algo'],'n_features':cfg['n_features'],
                              'F1_dir_ref':cfg['F1_dir']}
    model_proba_test[mid] = y_prob

    if (i+1) % 20 == 0 or i < 3:
        print(f"  [{i+1}/{n_total}] {mid[:50]} F1_dir={met['F1_dir']:.4f} ({time.time()-t0:.1f}s)")

print(f"\n[DONE] {len(trained_models)} modèles entraînés en {time.time()-t_total:.1f}s")


## Re-sélection après entraînement — corrélation des prédictions réelles

Maintenant qu'on a les prédictions sur le test, on peut calculer la vraie corrélation
entre les prédictions de chaque paire de modèles et éliminer les redondants.


In [ ]:
# =============================================================================
# RE-SÉLECTION PAR CORRÉLATION DES PRÉDICTIONS RÉELLES
# =============================================================================
print("=== Analyse de corrélation des prédictions ===")

# Matrice des directions prédites sur le test
dir_preds = {}
for mid, proba in model_proba_test.items():
    dir_preds[mid] = (proba[:,2]+proba[:,3] > 0.5).astype(int)

# Aligner sur la longueur minimale
min_len = min(len(v) for v in dir_preds.values())
dir_matrix = np.column_stack([v[:min_len] for v in dir_preds.values()])
model_ids  = list(dir_preds.keys())

# Matrice de corrélation
corr_matrix = np.corrcoef(dir_matrix.T)
corr_df = pd.DataFrame(corr_matrix, index=model_ids, columns=model_ids)

# Distribution des corrélations
upper = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
print(f"  Corrélation inter-modèles : mean={upper.mean():.3f} | "
      f"median={np.median(upper):.3f} | >0.85={( upper>0.85).mean():.1%}")

# Visualisation heatmap (top 30 modèles)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
if len(model_ids) <= 30:
    sns.heatmap(corr_df, ax=ax, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                xticklabels=False, yticklabels=False)
else:
    # Sous-ensemble des 30 meilleurs
    top30 = sorted(model_ids, key=lambda x: model_metrics[x].get('F1_dir',0), reverse=True)[:30]
    sns.heatmap(corr_df.loc[top30,top30], ax=ax, cmap='coolwarm', center=0,
                xticklabels=False, yticklabels=False)
ax.set_title('Corrélation inter-prédictions (top-30)')

ax = axes[1]
ax.hist(upper, bins=30, edgecolor='black', color='steelblue', alpha=0.7)
ax.axvline(0.85, color='red', linestyle='--', label='Seuil 0.85')
ax.set_title('Distribution corrélations inter-modèles')
ax.set_xlabel('Corrélation'); ax.legend()
plt.tight_layout(); plt.show()

# Re-sélection greedy par diversité sur les vraies prédictions
print("\nRe-sélection par diversité (corrélation prédictions réelles)...")
final_panel = select_diverse_panel(
    df_runs[df_runs['model_id'].isin(model_ids)],
    model_proba_test,
    max_corr=CONFIG['max_corr_pred'],
    top_n=200  # pas de limite stricte, laisser la diversité décider
)
print(f"Panel final après re-sélection : {len(final_panel)} modèles "
      f"(sur {len(model_ids)} entraînés)")

# Stats du panel final
df_final = pd.DataFrame([model_metrics[m] for m in final_panel if m in model_metrics])
print(f"\nF1_dir dans le panel final: mean={df_final['F1_dir'].mean():.4f} "
      f"max={df_final['F1_dir'].max():.4f} min={df_final['F1_dir'].min():.4f}")
print(df_final.groupby(['horizon','regime'])['model_id'].count().unstack(fill_value=0)
      if 'model_id' in df_final.columns else
      df_final.groupby(['horizon','regime']).size().unstack(fill_value=0))


In [ ]:
# =============================================================================
# STACKING OOF — Panel diversifié
# =============================================================================
META_TARGET_HORIZON = 5

print(f"=== STACKING — {len(final_panel)} modèles × 4 classes ===")
tgt_meta = targets_by_h[META_TARGET_HORIZON]['target']
tr_idx   = tgt_meta.index[tgt_meta.index <  pd.Timestamp(TEST_DATE)]
te_idx   = tgt_meta.index[tgt_meta.index >= pd.Timestamp(TEST_DATE)]
y_tr_dir = (tgt_meta.reindex(tr_idx).fillna(0).values >= 2).astype(int)
y_te_dir = (tgt_meta.reindex(te_idx).fillna(0).values >= 2).astype(int)
y_tr_4   = tgt_meta.reindex(tr_idx).fillna(0).values.astype(int)
y_te_4   = tgt_meta.reindex(te_idx).fillna(0).values.astype(int)

def get_oof_proba(model_id, n_folds=5):
    cfg   = df_runs[df_runs['model_id']==model_id].iloc[0]
    h     = int(cfg['horizon']); reg = str(cfg['regime'])
    algo  = str(cfg['algo'])
    feats = cfg['features']
    tgt_h = targets_by_h[h]['target']
    reg_s = targets_by_h[h]['regime']
    X_mat = build_feature_matrix(df_features.reindex(tgt_h.index), feats)
    if X_mat.empty: return None, None
    X_mat[TARGET_COL] = tgt_h
    df_tr2 = X_mat.loc[(X_mat.index < pd.Timestamp(TEST_DATE))].dropna(subset=[TARGET_COL])
    if reg != 'GLOBAL': df_tr2 = df_tr2.loc[reg_s.reindex(df_tr2.index)==reg]
    if len(df_tr2) < 50: return None, None
    fc    = [c for c in X_mat.columns if c!=TARGET_COL]
    X_t   = model_scalers[model_id].transform(df_tr2[fc].fillna(0).values)
    y_t   = df_tr2[TARGET_COL].values.astype(int)
    oof   = np.full((len(X_t),4), np.nan)
    tscv  = TimeSeriesSplit(n_splits=n_folds)
    for ti,vi in tscv.split(X_t):
        if len(vi)<5: continue
        try:
            sm = get_sampler(cfg.get('sampler','SMOTE'))
            Xr,yr = sm.fit_resample(X_t[ti],y_t[ti])
        except: Xr,yr = X_t[ti],y_t[ti]
        try:
            c = get_algo(algo,len(fc)); c.fit(Xr,yr)
            p = c.predict_proba(X_t[vi])
            if p.shape[1]<4:
                fp=np.zeros((len(p),4))
                for ci,cl in enumerate(c.classes_): fp[:,int(cl)]=p[:,ci]
                p=fp
            oof[vi]=p
        except: pass
    return oof, df_tr2.index

# Générer OOF
meta_tr_list, meta_te_list, used_ids = [], [], []
t0 = time.time()
for i, mid in enumerate(final_panel):
    if mid not in trained_models: continue
    oof, dates = get_oof_proba(mid)
    if oof is None: continue
    oof_df = pd.DataFrame(oof, index=dates,
                           columns=[f'{mid}_c{k}' for k in range(4)])
    meta_tr_list.append(oof_df.reindex(tr_idx).fillna(0.25))

    proba_te = model_proba_test[mid]
    cfg_row  = df_runs[df_runs['model_id']==mid].iloc[0]
    h = int(cfg_row['horizon']); reg = str(cfg_row['regime'])
    te_dates = targets_by_h[h]['target'].index[targets_by_h[h]['target'].index >= pd.Timestamp(TEST_DATE)]
    if reg!='GLOBAL': te_dates = te_dates[targets_by_h[h]['regime'].reindex(te_dates)==reg]
    te_df = pd.DataFrame(proba_te[:len(te_dates)], index=te_dates[:len(proba_te)],
                          columns=[f'{mid}_c{k}' for k in range(4)])
    meta_te_list.append(te_df.reindex(te_idx).fillna(0.25))
    used_ids.append(mid)
    if (i+1)%10==0: print(f"  OOF {i+1}/{len(final_panel)} ({time.time()-t0:.1f}s)")

meta_X_tr = pd.concat(meta_tr_list, axis=1).fillna(0.25) if meta_tr_list else pd.DataFrame()
meta_X_te = pd.concat(meta_te_list, axis=1).fillna(0.25) if meta_te_list else pd.DataFrame()
print(f"\nMéta-features: train={meta_X_tr.shape} test={meta_X_te.shape} ({time.time()-t0:.1f}s)")


In [ ]:
# =============================================================================
# MÉTA-MODÈLES + VOTE + RAPPORT
# =============================================================================
t0 = time.time()

# Vote pondéré baseline
def weighted_vote(meta_X_te, used_ids, model_metrics):
    su = np.zeros(len(meta_X_te)); sd = np.zeros(len(meta_X_te))
    for mid in used_ids:
        f1 = model_metrics.get(mid,{}).get('F1_dir',0.5)
        cu = [f'{mid}_c{k}' for k in [2,3] if f'{mid}_c{k}' in meta_X_te.columns]
        cd = [f'{mid}_c{k}' for k in [0,1] if f'{mid}_c{k}' in meta_X_te.columns]
        if cu: su += f1*meta_X_te[cu].sum(axis=1).values
        if cd: sd += f1*meta_X_te[cd].sum(axis=1).values
    return (su>sd).astype(int)

y_vote = weighted_vote(meta_X_te, used_ids, model_metrics)
f1_vote = f1_score(y_te_dir, y_vote, average='macro', zero_division=0)

# Méta-modèle direction
meta_dir = XGBClassifier(n_estimators=CONFIG['meta_n_estimators'],
                          max_depth=CONFIG['meta_max_depth'],
                          learning_rate=CONFIG['meta_lr'],
                          subsample=0.8, colsample_bytree=0.8,
                          eval_metric='logloss', random_state=SEED, n_jobs=-1)
meta_dir.fit(meta_X_tr.values, y_tr_dir)
y_pred_dir = meta_dir.predict(meta_X_te.values)
f1_meta_dir = f1_score(y_te_dir, y_pred_dir, average='macro', zero_division=0)
acc_meta_dir = accuracy_score(y_te_dir, y_pred_dir)

# Méta-modèle 4 classes
meta_4cls = XGBClassifier(n_estimators=CONFIG['meta_n_estimators'],
                           max_depth=CONFIG['meta_max_depth'],
                           learning_rate=CONFIG['meta_lr'],
                           subsample=0.8, colsample_bytree=0.8,
                           eval_metric='mlogloss', objective='multi:softprob',
                           random_state=SEED, n_jobs=-1)
meta_4cls.fit(meta_X_tr.values, y_tr_4)
y_pred_4cls = meta_4cls.predict(meta_X_te.values)
met_4cls = compute_metrics(y_te_4, y_pred_4cls)

# SHAP
try:
    expl = shap.TreeExplainer(meta_4cls)
    sv   = expl.shap_values(meta_X_te.values[:400])
    if isinstance(sv,list): arr=np.mean([np.abs(s) for s in sv],axis=0)
    elif np.array(sv).ndim==3: arr=np.abs(sv).mean(axis=2)
    else: arr=np.abs(sv)
    shap_scores = pd.Series(arr.mean(axis=0), index=meta_X_te.columns)
    top_shap = shap_scores.nlargest(30)
    print("\nTop-10 features SHAP méta-modèle:")
    for f,s in top_shap.head(10).items():
        mid_src = '_c'.join(f.split('_c')[:-1])
        info = model_metrics.get(mid_src,{})
        print(f"  {f[:55]:<55} SHAP={s:.5f} "
              f"(h={info.get('horizon','?')}j {info.get('regime','?')} "
              f"F1={info.get('F1_dir',0):.3f})")
except Exception as e: print(f"[WARN SHAP] {e}")

print(f"\n{'='*70}")
print(f"RAPPORT FINAL — {len(used_ids)} modèles dans le panel")
print(f"{'='*70}")
print(f"  Vote pondéré              F1_dir = {f1_vote:.4f}")
print(f"  Méta XGBoost direction    F1_dir = {f1_meta_dir:.4f}  Acc = {acc_meta_dir:.4f}")
print(f"  Méta XGBoost 4 classes    F1_dir = {met_4cls['F1_dir']:.4f}  "
      f"UP_FORT = {met_4cls.get('F1_UP_FORT',0):.4f}  "
      f"DOWN_FORT = {met_4cls.get('F1_DOWN_FORT',0):.4f}")
print(f"─"*70)
print(f"  Référence ML (LogReg h=5j GLOBAL) : F1_dir = 0.634")
print(f"  Stacking V1 (27 modèles)          : F1_dir = 0.596 (vote), 0.568 (méta)")

# Export
try:
    rows = [{'Model_ID':m,**model_metrics[m]} for m in used_ids if m in model_metrics]
    meta_res = [
        {'Model_ID':'Vote_Pondéré','F1_dir':f1_vote,'N_models':len(used_ids)},
        {'Model_ID':'Meta_XGB_dir','F1_dir':f1_meta_dir,'Acc_dir':acc_meta_dir,'N_models':len(used_ids)},
        {'Model_ID':'Meta_XGB_4cls',**met_4cls,'N_models':len(used_ids)},
    ]
    with pd.ExcelWriter('vix_stacking_v2_report.xlsx', engine='xlsxwriter') as w:
        pd.DataFrame(rows).sort_values('F1_dir',ascending=False).to_excel(w,'Panel_Models',index=False)
        pd.DataFrame(meta_res).to_excel(w,'Stacking_Results',index=False)
        top_shap.reset_index().rename(columns={'index':'Feature',0:'SHAP'}).to_excel(w,'SHAP_Meta',index=False)
    print("\n[SAVE] vix_stacking_v2_report.xlsx")
except Exception as e: print(f"[WARN Export] {e}")
print("[NOTE] Aucun modèle enregistré — validation explicite requise.")
